In [1]:
# ============================================================
# CELL 1 — Install dependencies & load secrets
# ============================================================

# | Secret name | Value |
# |---|---|
# | `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
# | `DB_PORT` | usually `5432` |
# | `DB_NAME` | your database name |
# | `DB_USER` | your database user |
# | `DB_PASSWORD` | your database password |
# | `JWT_SECRET` | any long random string (generate one in the next cell) |
# | `SMTP_EMAIL` | your Gmail address |
# | `SMTP_APP_PASSWORD` | 16-character Gmail App Password (not your real password) |
# | `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |

!pip install -q streamlit psycopg2-binary PyJWT bcrypt \
    python-dotenv email-validator pyngrok \
    fastapi uvicorn python-multipart requests \
    langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib reportlab \
    transformers accelerate torch stopwordsiso deepface opencv-python-headless

!python -m spacy download xx_sent_ud_sm -q


from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━

In [2]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv

# Ensure environment variables are loaded, overriding any existing ones
load_dotenv(override=True)

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    # Print the DB_HOST being used for debugging
    print(f"DEBUG: DB_HOST being used by db.py: {os.getenv('DB_HOST')}")
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE,
            role VARCHAR(20) NOT NULL DEFAULT 'employee')""")
        cur.execute("""ALTER TABLE users ADD COLUMN IF NOT EXISTS role VARCHAR(20) NOT NULL DEFAULT 'employee'""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")

        cur.execute("""CREATE TABLE IF NOT EXISTS mood_logs (
            id SERIAL PRIMARY KEY,
            user_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
            mood_date DATE NOT NULL DEFAULT CURRENT_DATE,
            sentiment VARCHAR(20),
            emotion VARCHAR(30),
            compound_score REAL,
            confidence REAL,
            positive_score REAL,
            negative_score REAL,
            neutral_score REAL,
            detected_language VARCHAR(80),
            cleaned_text TEXT,
            journal_text TEXT,
            source VARCHAR(10) NOT NULL DEFAULT 'manual',
            created_at TIMESTAMP NOT NULL DEFAULT NOW())""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS source VARCHAR(10) NOT NULL DEFAULT 'manual'""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS confidence REAL""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS positive_score REAL""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS negative_score REAL""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS neutral_score REAL""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS detected_language VARCHAR(80)""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS cleaned_text TEXT""")
        cur.execute("""CREATE TABLE IF NOT EXISTS daily_wellness (
            id SERIAL PRIMARY KEY,
            user_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
            wellness_date DATE NOT NULL DEFAULT CURRENT_DATE,
            stress_level REAL,
            sleep_hours REAL,
            workload VARCHAR(20),
            created_at TIMESTAMP NOT NULL DEFAULT NOW(),
            UNIQUE(user_id, wellness_date)
        )""")
        cur.execute("""CREATE INDEX IF NOT EXISTS idx_daily_wellness_user_date ON daily_wellness(user_id, wellness_date)""")
        cur.execute("""CREATE INDEX IF NOT EXISTS idx_mood_logs_user_date
            ON mood_logs(user_id, mood_date)""")


MOOD_LABELS = ["Amazing", "Happy", "Normal", "Sad", "Angry"]

NLP_TO_MOOD_LABEL = {
    "Positive": "Happy",
    "Neutral": "Normal",
    "Negative": "Sad",
}


def save_manual_mood(user_id, mood_label):
    """Employee taps an emoji on the 'How Do You Feel?' picker — saves
    immediately (with the current date+time via created_at), no NLP involved."""
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, source)
               VALUES (%s, %s, 'manual')""",
            (user_id, mood_label),
        )

def save_mood_log(user_id, sentiment, emotion, compound_score, journal_text, confidence=None,
                  positive_score=None, negative_score=None, neutral_score=None,
                  detected_language=None, cleaned_text=None):
    """Store the NLP result exactly as returned by the existing pipeline.
    NLP is not rerun by the weekly report; these stored values are reused."""
    mood_label = NLP_TO_MOOD_LABEL.get(sentiment, "Normal")
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs
               (user_id, sentiment, emotion, compound_score, confidence,
                positive_score, negative_score, neutral_score, detected_language,
                cleaned_text, journal_text, source)
               VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, 'nlp')""",
            (user_id, mood_label, emotion, compound_score, confidence,
             positive_score, negative_score, neutral_score, detected_language,
             cleaned_text, journal_text),
        )

def save_daily_wellness(user_id, wellness_date, stress_level=None, sleep_hours=None, workload=None):
    """Upsert non-journal daily wellness measurements without creating journal duplicates."""
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO daily_wellness
               (user_id, wellness_date, stress_level, sleep_hours, workload)
               VALUES (%s, %s, %s, %s, %s)
               ON CONFLICT (user_id, wellness_date) DO UPDATE SET
                 stress_level = EXCLUDED.stress_level,
                 sleep_hours = EXCLUDED.sleep_hours,
                 workload = EXCLUDED.workload,
                 created_at = NOW()""",
            (user_id, wellness_date, stress_level, sleep_hours, workload),
        )

def get_daily_wellness_range(user_id, start_date, end_date):
    with cursor() as cur:
        cur.execute(
            """SELECT wellness_date, stress_level, sleep_hours, workload, created_at
               FROM daily_wellness
               WHERE user_id = %s AND wellness_date BETWEEN %s AND %s
               ORDER BY wellness_date""",
            (user_id, start_date, end_date),
        )
        return cur.fetchall()

def get_mood_logs_for_month(user_id, year, month):
    """Returns one row per day for a given user/month, latest entry per day.
    Used by the Home tab's calendar grid."""
    with cursor() as cur:
        cur.execute(
            """SELECT DISTINCT ON (mood_date) mood_date, sentiment, emotion, compound_score, confidence, created_at
               FROM mood_logs
               WHERE user_id = %s
                 AND EXTRACT(YEAR FROM mood_date) = %s
                 AND EXTRACT(MONTH FROM mood_date) = %s
               ORDER BY mood_date, created_at DESC""",
            (user_id, year, month),
        )
        return cur.fetchall()

def get_user_mood_history(user_id, limit=200):
    """Full history for ONE user, newest first — every field including the
    exact created_at timestamp and journal_text. Powers both the Journal
    tab's 'past entries' list and the personal Dashboard tab's charts."""
    with cursor() as cur:
        cur.execute(
            """SELECT mood_date, sentiment, emotion, compound_score, confidence, positive_score, negative_score, neutral_score, detected_language, cleaned_text, journal_text, source, created_at
               FROM mood_logs
               WHERE user_id = %s
               ORDER BY created_at DESC
               LIMIT %s""",
            (user_id, limit),
        )
        return cur.fetchall()

def get_all_employee_mood_logs(limit_days=30):
    """For managers: every employee's mood entries from the last N days,
    joined with username for display."""
    with cursor() as cur:
        cur.execute(
            """SELECT u.username, u.email, m.mood_date, m.sentiment, m.emotion, m.compound_score, m.confidence, m.created_at
               FROM mood_logs m
               JOIN users u ON u.id = m.user_id
               WHERE u.role = 'employee'
                 AND m.mood_date >= CURRENT_DATE - (%s || ' days')::interval
               ORDER BY m.mood_date DESC, u.username""",
            (limit_days,),
        )
        return cur.fetchall()

def get_latest_mood_per_employee():
    """For managers: each employee's single most recent mood entry."""
    with cursor() as cur:
        cur.execute(
            """SELECT DISTINCT ON (u.id) u.username, u.email, m.mood_date, m.sentiment, m.emotion, m.confidence, m.created_at
               FROM users u
               JOIN mood_logs m ON m.user_id = u.id
               WHERE u.role = 'employee'
               ORDER BY u.id, m.created_at DESC"""
        )
        return cur.fetchall()

Writing db.py


In [3]:
# ============================================================
# CELL 3 — auth.py  (UNCHANGED from your original)
# ============================================================
%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())

def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "role": user.get("role", "employee"),
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw, role="employee"):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash,role) VALUES (%s,%s,%s,%s)",
                    (username, email, hash_pw(pw), role))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True

Writing auth.py


In [4]:
# ============================================================
# CELL 4 — email_utils.py  (UNCHANGED from your original)
# ============================================================
%%writefile email_utils.py
import os, smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    subject = "Your Verification Code" if purpose == "signup" else "Your Password Reset Code"
    msg = MIMEText(f"Your code is: {code}\nExpires in 10 minutes.")
    msg["From"], msg["To"], msg["Subject"] = EMAIL, to_email, subject
    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)

Writing email_utils.py


In [5]:
%%writefile welcome_image.py
# Auto-generated: base64-encoded welcome/login illustration
# (portrait crop, 700x875, JPEG) - embedded so it survives Colab runtime resets
WELCOME_IMAGE_B64 = "..."

Writing welcome_image.py


In [6]:
%%writefile weekly_report.py
from collections import Counter
from io import BytesIO
from statistics import mean, pstdev
from datetime import timedelta

DEFAULT_WEEKLY_WEIGHTS = {
    "Daily Mood": 20.0,
    "Journal Emotion": 20.0,
    "Sentiment": 15.0,
    "Stress": 15.0,
    "Sleep": 10.0,
    "Workload": 10.0,
    "Journal Consistency": 10.0,
}

MOOD_SCORE = {"Amazing": 100.0, "Happy": 85.0, "Normal": 65.0, "Sad": 35.0, "Angry": 15.0}
EMOTION_SCORE = {"Happy": 100.0, "Joy": 100.0, "Excited": 95.0, "Calm": 90.0,
                 "Neutral": 65.0, "Sad": 35.0, "Stress": 30.0, "Angry": 20.0, "Fear": 25.0}
WORKLOAD_SCORE = {"Low": 100.0, "Medium": 70.0, "High": 35.0}


def _clamp(v, lo=0.0, hi=100.0):
    return max(lo, min(hi, float(v)))


def _sentiment_score(day):
    compound = day.get("compound_score")
    if compound is not None:
        return _clamp((float(compound) + 1.0) * 50.0)
    return MOOD_SCORE.get(day.get("sentiment"), None)


def _sleep_score(hours):
    if hours is None:
        return None
    h = float(hours)
    if 7.0 <= h <= 9.0:
        return 100.0
    if h < 7.0:
        return _clamp(100.0 - (7.0 - h) * 20.0)
    return _clamp(100.0 - (h - 9.0) * 12.5, 70.0, 100.0)


def daily_component_scores(day):
    scores = {}
    if day.get("mood"):
        scores["Daily Mood"] = MOOD_SCORE.get(day["mood"], 65.0)
    if day.get("emotion"):
        scores["Journal Emotion"] = EMOTION_SCORE.get(day["emotion"], 65.0)
    if day.get("compound_score") is not None or day.get("sentiment"):
        scores["Sentiment"] = _sentiment_score(day)
    if day.get("stress_level") is not None:
        scores["Stress"] = _clamp((10.0 - float(day["stress_level"])) * 10.0)
    if day.get("sleep_hours") is not None:
        scores["Sleep"] = _sleep_score(day["sleep_hours"])
    if day.get("workload"):
        scores["Workload"] = WORKLOAD_SCORE.get(str(day["workload"]).title(), 70.0)
    return scores


def calculate_daily_score(day, weights=None):
    weights = weights or DEFAULT_WEEKLY_WEIGHTS
    scores = daily_component_scores(day)
    usable = {k: weights.get(k, 0.0) for k in scores if scores.get(k) is not None and weights.get(k, 0.0) > 0}
    if not usable:
        return None
    total_w = sum(usable.values())
    return round(sum(scores[k] * usable[k] for k in usable) / total_w, 1)


def build_weekly_report_data(user_id, end_date, get_user_mood_history, get_daily_wellness_range):
    start_date = end_date - timedelta(days=6)
    mood_rows = [r for r in get_user_mood_history(user_id, limit=5000)
                 if start_date <= r["mood_date"] <= end_date]
    wellness_rows = get_daily_wellness_range(user_id, start_date, end_date)
    wellness_by_date = {r["wellness_date"]: r for r in wellness_rows}
    days = []
    for offset in range(7):
        d = start_date + timedelta(days=offset)
        rows = [r for r in mood_rows if r["mood_date"] == d]
        manual_rows = [r for r in rows if r.get("source") == "manual"]
        nlp_rows = [r for r in rows if r.get("source") == "nlp" or r.get("emotion")]
        manual_latest = manual_rows[0] if manual_rows else None
        nlp_latest = nlp_rows[0] if nlp_rows else None
        latest = nlp_latest or (rows[0] if rows else None)
        journal_rows = [r for r in nlp_rows if r.get("journal_text") and r["journal_text"].strip()]
        w = wellness_by_date.get(d) or {}
        day = {
            "date": d,
            # Prefer the employee's explicitly selected mood over an NLP-derived mood.
            "mood": manual_latest.get("sentiment") if manual_latest else (latest.get("sentiment") if latest else None),
            "emotion": nlp_latest.get("emotion") if nlp_latest else None,
            "emotion_confidence": nlp_latest.get("confidence") if nlp_latest else None,
            "sentiment": nlp_latest.get("sentiment") if nlp_latest else None,
            "compound_score": nlp_latest.get("compound_score") if nlp_latest else None,
            "positive_score": nlp_latest.get("positive_score") if nlp_latest else None,
            "negative_score": nlp_latest.get("negative_score") if nlp_latest else None,
            "neutral_score": nlp_latest.get("neutral_score") if nlp_latest else None,
            "journal_text": journal_rows[0].get("journal_text") if journal_rows else None,
            "stress_level": w.get("stress_level"),
            "sleep_hours": w.get("sleep_hours"),
            "workload": w.get("workload"),
            "has_wellness_data": bool(latest or w),
            "has_journal": bool(journal_rows),
        }
        days.append(day)
    return {"start_date": start_date, "end_date": end_date, "days": days}


def aggregate_week(days, weights):
    available_days = [d for d in days if d["has_wellness_data"]]
    coverage = len(available_days) / 7.0 * 100.0
    journal_days = sum(1 for d in days if d["has_journal"])
    consistency = journal_days / 7.0 * 100.0

    component_values = {k: [] for k in weights}
    daily_scores = []
    for d in days:
        scores = daily_component_scores(d)
        d["component_scores"] = scores
        d["daily_score"] = calculate_daily_score(d, weights)
        if d["daily_score"] is not None:
            daily_scores.append((d["date"], d["daily_score"]))
        for k, v in scores.items():
            component_values.setdefault(k, []).append(v)
    component_values["Journal Consistency"] = [consistency]

    usable = {k: mean(v) for k, v in component_values.items() if v}
    active_weights = {k: weights.get(k, 0.0) for k in usable if weights.get(k, 0.0) > 0}
    weight_sum = sum(active_weights.values())
    weekly_score = round(sum(usable[k] * active_weights[k] for k in active_weights) / weight_sum, 1) if weight_sum else None

    stress_values = [float(d["stress_level"]) for d in days if d["stress_level"] is not None]
    sleep_values = [float(d["sleep_hours"]) for d in days if d["sleep_hours"] is not None]
    workload_values = [d["workload"] for d in days if d["workload"]]
    mood_values = [d["mood"] for d in days if d["mood"]]
    emotion_values = [d["emotion"] for d in days if d["emotion"]]
    positive_scores = [float(d["positive_score"]) for d in days if d["positive_score"] is not None]
    negative_scores = [float(d["negative_score"]) for d in days if d["negative_score"] is not None]
    neutral_scores = [float(d["neutral_score"]) for d in days if d["neutral_score"] is not None]
    compounds = [float(d["compound_score"]) for d in days if d["compound_score"] is not None]
    conf_values = [float(d["emotion_confidence"]) for d in days if d["emotion_confidence"] is not None]

    trend_stress = None
    if len(stress_values) >= 2:
        first = next(d["stress_level"] for d in days if d["stress_level"] is not None)
        last = next(d["stress_level"] for d in reversed(days) if d["stress_level"] is not None)
        trend_stress = "increasing" if last > first + 0.5 else "decreasing" if last < first - 0.5 else "stable"

    workload_counts = dict(Counter(str(x).title() for x in workload_values))
    emotion_counts = dict(Counter(emotion_values))
    mood_counts = dict(Counter(mood_values))
    sentiment_counts = dict(Counter(d["sentiment"] for d in days if d.get("sentiment")))

    return {
        "coverage_days": len(available_days), "coverage_pct": round(coverage, 2),
        "journal_days": journal_days, "journal_consistency": round(consistency, 2),
        "weekly_score": weekly_score, "daily_scores": daily_scores,
        "component_averages": usable,
        "avg_stress": mean(stress_values) if stress_values else None,
        "min_stress": min(stress_values) if stress_values else None,
        "max_stress": max(stress_values) if stress_values else None,
        "stress_trend": trend_stress,
        "avg_sleep": mean(sleep_values) if sleep_values else None,
        "min_sleep": min(sleep_values) if sleep_values else None,
        "max_sleep": max(sleep_values) if sleep_values else None,
        "sleep_consistency": round(100.0 - (pstdev(sleep_values) * 20.0), 1) if len(sleep_values) > 1 else (100.0 if sleep_values else None),
        "avg_workload": (sum(WORKLOAD_SCORE.get(str(x).title(), 70.0) for x in workload_values) / len(workload_values)) if workload_values else None,
        "high_workload_days": sum(1 for x in workload_values if str(x).title() == "High"),
        "workload_counts": workload_counts,
        "mood_counts": mood_counts,
        "emotion_counts": emotion_counts,
        "sentiment_counts": sentiment_counts,
        "most_common_mood": Counter(mood_values).most_common(1)[0][0] if mood_values else None,
        "most_common_emotion": Counter(emotion_values).most_common(1)[0][0] if emotion_values else None,
        "positive_emotion_days": sum(v for k, v in emotion_counts.items() if k in {"Happy", "Joy", "Excited", "Calm"}),
        "negative_emotion_days": sum(v for k, v in emotion_counts.items() if k in {"Sad", "Stress", "Angry", "Fear"}),
        "avg_emotion_confidence": mean(conf_values) if conf_values else None,
        "avg_positive": mean(positive_scores) if positive_scores else None,
        "avg_negative": mean(negative_scores) if negative_scores else None,
        "avg_neutral": mean(neutral_scores) if neutral_scores else None,
        "avg_compound": mean(compounds) if compounds else None,
    }


def generate_weekly_summary(stats):
    score = stats.get("weekly_score")
    score_text = f"{score:.0f}/100" if score is not None else "not available"
    mood = stats.get("most_common_mood") or "not enough mood data"
    emotion = stats.get("most_common_emotion") or "not enough emotion data"
    stress = f"{stats['avg_stress']:.1f}/10" if stats.get("avg_stress") is not None else "not available"
    sleep = f"{stats['avg_sleep']:.1f} hours" if stats.get("avg_sleep") is not None else "not available"
    compound = f"{stats['avg_compound']:.2f}" if stats.get("avg_compound") is not None else "not available"
    return (f"Your weekly wellness score is {score_text}, based on {stats['coverage_days']}/7 days ({stats['coverage_pct']:.2f}%) of actual stored wellness data. "
            f"Your most common mood was {mood}, and your most common detected emotion was {emotion}. "
            f"Average stress was {stress}, average sleep was {sleep}, and average compound sentiment was {compound}. "
            "Missing measurements were not treated as zero; only available components contributed to the score and their configured weights were redistributed.")


def recommendations(stats):
    rec = []
    if stats.get("avg_stress") is not None and stats["avg_stress"] >= 7:
        rec += ["Take regular short breaks during demanding periods.", "Prioritize urgent tasks and discuss sustained workload pressure when needed."]
    if stats.get("avg_sleep") is not None and stats["avg_sleep"] < 6:
        rec += ["Try to maintain a consistent sleep schedule and aim for sufficient nightly sleep."]
    if stats.get("high_workload_days", 0) >= 3:
        rec += ["Break large tasks into smaller steps and review workload distribution."]
    if stats.get("journal_consistency", 0) < 50:
        rec += ["Recording mood and wellness details more consistently will improve future weekly insights."]
    if not rec:
        rec = ["Continue the habits that are supporting your current wellness pattern.", "Maintain a healthy balance between focused work, recovery, sleep, and regular check-ins."]
    return rec


def achievements(stats):
    out = []
    if stats.get("coverage_days") == 7:
        out.append("🏆 7-Day Wellness Tracker")
    if stats.get("most_common_mood") in {"Amazing", "Happy"} or stats.get("positive_emotion_days", 0) > stats.get("negative_emotion_days", 0):
        out.append("😊 Positive Week")
    if stats.get("journal_days", 0) >= 5:
        out.append("💪 Consistent Journal Writer")
    if (stats.get("avg_stress") is not None and stats["avg_stress"] <= 4.5 and
        stats.get("avg_sleep") is not None and stats["avg_sleep"] >= 7 and
        stats.get("most_common_mood") in {"Amazing", "Happy", "Normal"}):
        out.append("🌟 Healthy Work-Life Balance")
    return out or ["🌱 Keep Building Your Wellness Record"]


def build_weekly_pdf(username, email, report, stats, summary, recs, awards, figures):
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

    buf = BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4, rightMargin=32, leftMargin=32, topMargin=32, bottomMargin=32)
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="ReportTitle", parent=styles["Title"], alignment=TA_CENTER, fontSize=22, leading=27, spaceAfter=10))
    styles.add(ParagraphStyle(name="Section", parent=styles["Heading2"], fontSize=14, leading=18, spaceBefore=10, spaceAfter=7))
    styles.add(ParagraphStyle(name="Small", parent=styles["BodyText"], fontSize=8.5, leading=11))
    story = [Paragraph("WEEKLY WELLNESS REPORT", styles["ReportTitle"]),
             Paragraph(f"Employee: {username} · {email}", styles["BodyText"]),
             Paragraph(f"Report Period: {report['start_date']} to {report['end_date']}", styles["BodyText"]), Spacer(1, 12)]
    score = f"{stats['weekly_score']:.0f} / 100" if stats.get("weekly_score") is not None else "Not available"
    overview = [
        ["Wellness Score", score], ["Data Coverage", f"{stats['coverage_days']} / 7 days ({stats['coverage_pct']:.2f}%)"],
        ["Journal Consistency", f"{stats['journal_days']} / 7 days ({stats['journal_consistency']:.2f}%)"],
        ["Average Stress", f"{stats['avg_stress']:.1f} / 10" if stats.get('avg_stress') is not None else "Unavailable"],
        ["Average Sleep", f"{stats['avg_sleep']:.1f} hrs" if stats.get('avg_sleep') is not None else "Unavailable"],
        ["Most Common Mood", stats.get("most_common_mood") or "Unavailable"],
        ["Most Common Emotion", stats.get("most_common_emotion") or "Unavailable"],
    ]
    t = Table(overview, colWidths=[2.2*inch, 3.8*inch])
    t.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,-1), colors.whitesmoke), ("GRID", (0,0), (-1,-1), .5, colors.lightgrey), ("FONTNAME", (0,0), (0,-1), "Helvetica-Bold"), ("VALIGN", (0,0), (-1,-1), "MIDDLE"), ("PADDING", (0,0), (-1,-1), 7)]))
    story += [t, Paragraph("AI Weekly Summary", styles["Section"]), Paragraph(summary, styles["BodyText"])]

    story.append(Paragraph("Daily Wellness Scores", styles["Section"]))
    rows = [["Date", "Mood", "Emotion", "Stress", "Sleep", "Workload", "Score"]]
    for d in report["days"]:
        rows.append([str(d["date"]), d.get("mood") or "—", d.get("emotion") or "—",
                     f"{d['stress_level']:.1f}" if d.get("stress_level") is not None else "—",
                     f"{d['sleep_hours']:.1f}" if d.get("sleep_hours") is not None else "—",
                     d.get("workload") or "—", f"{d['daily_score']:.1f}" if d.get("daily_score") is not None else "—"])
    dt = Table(rows, repeatRows=1, colWidths=[.75*inch, .75*inch, .85*inch, .55*inch, .55*inch, .7*inch, .55*inch])
    dt.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,0), colors.HexColor("#6B21A8")), ("TEXTCOLOR", (0,0), (-1,0), colors.white), ("GRID", (0,0), (-1,-1), .3, colors.lightgrey), ("FONTSIZE", (0,0), (-1,-1), 7), ("ALIGN", (0,0), (-1,-1), "CENTER"), ("VALIGN", (0,0), (-1,-1), "MIDDLE")]))
    story.append(dt)

    story.append(Paragraph("Analyses", styles["Section"]))
    analysis_lines = [
        f"Stress: average {stats['avg_stress']:.1f}/10, minimum {stats['min_stress']:.1f}, maximum {stats['max_stress']:.1f}, trend {stats['stress_trend'] or 'unavailable'}." if stats.get('avg_stress') is not None else "Stress: unavailable.",
        f"Sleep: average {stats['avg_sleep']:.1f} hours, minimum {stats['min_sleep']:.1f}, maximum {stats['max_sleep']:.1f}." if stats.get('avg_sleep') is not None else "Sleep: unavailable.",
        f"Workload: {stats.get('workload_counts') or 'unavailable'}; high-workload days: {stats.get('high_workload_days', 0)}.",
        f"Emotion confidence: {stats['avg_emotion_confidence']:.1%}." if stats.get('avg_emotion_confidence') is not None else "Emotion confidence: unavailable.",
        f"Sentiment averages: positive {stats['avg_positive']:.2f}, negative {stats['avg_negative']:.2f}, neutral {stats['avg_neutral']:.2f}, compound {stats['avg_compound']:.2f}." if stats.get('avg_positive') is not None else "Stored positive/negative/neutral sentiment scores are unavailable for this period; compound sentiment is used where stored.",
    ]
    for line in analysis_lines:
        story.append(Paragraph("• " + line, styles["BodyText"]))

    for title, fig in figures:
        if fig is None:
            continue
        img_buf = BytesIO()
        fig.savefig(img_buf, format="png", dpi=140, bbox_inches="tight")
        img_buf.seek(0)
        story += [Paragraph(title, styles["Section"]), Image(img_buf, width=6.1*inch, height=3.0*inch)]

    story.append(PageBreak())
    story.append(Paragraph("Personalized Recommendations", styles["Section"]))
    for r in recs:
        story.append(Paragraph("• " + r, styles["BodyText"]))
    story.append(Paragraph("Achievements", styles["Section"]))
    for a in awards:
        story.append(Paragraph(a, styles["BodyText"]))
    story.append(Spacer(1, 12))
    story.append(Paragraph("This report is based only on wellness information stored for the selected 7-day period. Missing values are shown as unavailable and are not treated as zero.", styles["Small"]))
    doc.build(story)
    return buf.getvalue()


Writing weekly_report.py


In [7]:
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install opencv-python-headless==4.10.0.84

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.2 MB/s eta 0:00:00


In [8]:
import cv2

print("OpenCV version:", cv2.__version__)
print("CascadeClassifier:", hasattr(cv2, "CascadeClassifier"))

OpenCV version: 4.10.0
CascadeClassifier: True


In [9]:
!streamlit run /content/app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: /content/app.py


In [10]:
# ============================================================
# CELL 5 — app.py  (MAXIMUM ANIMATION — floating blobs, shimmer
# sweeps, pulsing glows, staggered pop-ins, sparkles, bounce)
# ============================================================
%%writefile app.py
import os, re, random, calendar
from datetime import date, datetime
import requests, streamlit as st
import matplotlib.pyplot as plt
from db import (init_db, save_mood_log, save_manual_mood, MOOD_LABELS,
                 get_mood_logs_for_month, get_user_mood_history,
                 get_all_employee_mood_logs, get_latest_mood_per_employee,
                 save_daily_wellness, get_daily_wellness_range)
from auth import (make_token, read_token, get_user, username_taken, create_user,
                   verify_user, set_password, check_pw, new_otp, save_otp, check_otp)
from email_utils import send_otp
from welcome_image import WELCOME_IMAGE_B64
from weekly_report import (DEFAULT_WEEKLY_WEIGHTS, build_weekly_report_data, aggregate_week,
                           generate_weekly_summary, recommendations, achievements, build_weekly_pdf)

st.set_page_config(page_title="MoodMentor", page_icon="🧠", layout="wide")

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

# ---- Vivid multi-hue palette ----
BRAND_GREEN = "#9F7AEA"        # primary purple (var name kept for compatibility)
BRAND_GREEN_DARK = "#805AD5"
PRIMARY = BRAND_GREEN
PRIMARY_DARK = BRAND_GREEN_DARK
PINK = "#EC4899"
CYAN = "#06B6D4"
GREEN = "#10B981"
AMBER = "#F59E0B"
CORAL = "#F97066"
TITLE_COLOR = "#6B21A8"
INK = "#1E1B2E"
MUTED = "#6B7280"
GLASS_BG = "rgba(255, 255, 255, 0.62)"
GLASS_BORDER = "rgba(255, 255, 255, 0.9)"
BTN_GRADIENT = "linear-gradient(135deg, #EC4899 0%, #9F7AEA 50%, #6366F1 100%)"
BTN_HOVER = "linear-gradient(135deg, #DB2777 0%, #805AD5 50%, #4F46E5 100%)"

QUOTES = [
    "Small steps lead to big changes.",
    "Every day is a fresh start.",
    "Breathe in peace, exhale stress.",
    "You are doing better than you think.",
    "Your mental health is a priority.",
]

MOOD_STYLE = {
    "Amazing": {"emoji": "🤩", "color": "#10B981", "bg": "#D1FAE5", "border": "#10B981", "glow": "rgba(16,185,129,0.45)"},
    "Happy":   {"emoji": "😀", "color": "#EC4899", "bg": "#FCE7F3", "border": "#EC4899", "glow": "rgba(236,72,153,0.45)"},
    "Normal":  {"emoji": "😐", "color": "#06B6D4", "bg": "#CFFAFE", "border": "#06B6D4", "glow": "rgba(6,182,212,0.45)"},
    "Sad":     {"emoji": "🙁", "color": "#F59E0B", "bg": "#FEF3C7", "border": "#F59E0B", "glow": "rgba(245,158,11,0.45)"},
    "Angry":   {"emoji": "🤬", "color": "#EF4444", "bg": "#FEE2E2", "border": "#EF4444", "glow": "rgba(239,68,68,0.45)"},
}
def style_for(label):
    return MOOD_STYLE.get(label, {"emoji": "⬜", "color": "#bdbdbd", "bg": "#F7FAFC", "border": "#F7FAFC", "glow": "transparent"})

MOOD_TO_NUM = {"Amazing": 2, "Happy": 1, "Normal": 0, "Sad": -1, "Angry": -2}

def inject_css():
    st.markdown(f"""
    <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700;800;900&display=swap" rel="stylesheet">
    <style>
        html, body, [class*="css"] {{ font-family: 'Poppins', sans-serif; }}

        @media (prefers-reduced-motion: reduce) {{
            * {{ animation: none !important; transition: none !important; }}
        }}

        /* ---- Animated moving-gradient app background ---- */
        .stApp {{
            background:
                radial-gradient(900px 500px at 5% 0%, rgba(236,72,153,0.20), transparent 55%),
                radial-gradient(900px 500px at 95% 10%, rgba(99,102,241,0.22), transparent 55%),
                radial-gradient(800px 500px at 50% 100%, rgba(6,182,212,0.18), transparent 55%),
                linear-gradient(160deg, #FDF4FF 0%, #F0F4FF 45%, #ECFEFF 100%);
            background-size: 140% 140%, 140% 140%, 140% 140%, 200% 200%;
            animation: bgFloat 16s ease-in-out infinite alternate;
        }}
        @keyframes bgFloat {{
            0%   {{ background-position: 0% 0%, 100% 0%, 50% 100%, 0% 0%; }}
            100% {{ background-position: 10% 10%, 90% 15%, 55% 90%, 100% 100%; }}
        }}
        #MainMenu, footer {{visibility: hidden;}}

        ::-webkit-scrollbar {{ width: 10px; height: 10px; }}
        ::-webkit-scrollbar-track {{ background: transparent; }}
        ::-webkit-scrollbar-thumb {{
            background: linear-gradient(180deg, {PINK}, {PRIMARY}, {CYAN});
            border-radius: 10px;
        }}

        /* ---- Floating decorative blobs (behind glass panels) ---- */
        @keyframes blobFloat {{
            0%   {{ transform: translate(0,0) scale(1) rotate(0deg); }}
            33%  {{ transform: translate(20px,-25px) scale(1.08) rotate(8deg); }}
            66%  {{ transform: translate(-15px,15px) scale(0.95) rotate(-6deg); }}
            100% {{ transform: translate(0,0) scale(1) rotate(0deg); }}
        }}

        /* ---- Sparkle particles ---- */
        @keyframes sparkleFloat {{
            0%   {{ transform: translateY(0) scale(0.8); opacity: 0.2; }}
            50%  {{ transform: translateY(-18px) scale(1.15); opacity: 1; }}
            100% {{ transform: translateY(0) scale(0.8); opacity: 0.2; }}
        }}
        .sparkle {{
            position: absolute; font-size: 1.1rem; pointer-events: none;
            animation: sparkleFloat 3.5s ease-in-out infinite;
            filter: drop-shadow(0 0 6px rgba(236,72,153,0.6));
        }}

        /* ---- Pop-in / staggered entrance ---- */
        @keyframes popIn {{
            0%   {{ opacity: 0; transform: translateY(16px) scale(0.94); }}
            60%  {{ opacity: 1; transform: translateY(-3px) scale(1.015); }}
            100% {{ opacity: 1; transform: translateY(0) scale(1); }}
        }}
        @keyframes fadeInUp {{
            from {{ opacity: 0; transform: translateY(10px); }}
            to   {{ opacity: 1; transform: translateY(0); }}
        }}

        /* ---- Pulsing glow ring ---- */
        @keyframes glowPulse {{
            0%, 100% {{ box-shadow: 0 22px 50px -18px rgba(99, 102, 241, 0.28); }}
            50%      {{ box-shadow: 0 26px 60px -16px rgba(236, 72, 153, 0.4); }}
        }}

        /* ---- Sidebar ---- */
        section[data-testid="stSidebar"] {{
            background: linear-gradient(180deg, rgba(255,255,255,0.85) 0%, rgba(253,244,255,0.85) 100%);
            backdrop-filter: blur(24px);
            -webkit-backdrop-filter: blur(24px);
            border-right: 1px solid {GLASS_BORDER};
        }}
        section[data-testid="stSidebar"] .stRadio > label {{ font-weight: 700; color: {INK}; }}
        section[data-testid="stSidebar"] div[role="radiogroup"] label {{
            padding: 11px 15px; border-radius: 14px; margin-bottom: 5px;
            transition: all 200ms ease-out;
        }}
        section[data-testid="stSidebar"] div[role="radiogroup"] label:hover {{
            background: linear-gradient(90deg, rgba(236,72,153,0.16), rgba(99,102,241,0.16));
            transform: translateX(5px) scale(1.02);
        }}

        /* ---- Glow-ring gradient border card, pulsing + pop-in ---- */
        .mm-card {{
            position: relative;
            background: {GLASS_BG};
            backdrop-filter: blur(20px);
            -webkit-backdrop-filter: blur(20px);
            border-radius: 26px; padding: 28px 30px;
            border: 1.5px solid transparent;
            background-clip: padding-box;
            margin-bottom: 22px;
            animation: popIn 550ms cubic-bezier(0.22,1,0.36,1), glowPulse 5s ease-in-out infinite 600ms;
        }}
        .mm-card::before {{
            content: ""; position: absolute; inset: -1.5px; border-radius: 27px; z-index: -1;
            background: linear-gradient(135deg, rgba(236,72,153,0.55), rgba(99,102,241,0.55), rgba(6,182,212,0.55));
            opacity: 0.6;
            background-size: 200% 200%;
            animation: gradientShift 6s ease infinite;
        }}
        @keyframes gradientShift {{
            0%   {{ background-position: 0% 50%; }}
            50%  {{ background-position: 100% 50%; }}
            100% {{ background-position: 0% 50%; }}
        }}
        .mm-card h4, .mm-card h3 {{ margin-top: 0; color: {TITLE_COLOR}; font-weight: 700; }}

        /* ---- Metric tiles: staggered bounce-in + hover lift ---- */
        .mm-metric {{
            background: {GLASS_BG};
            backdrop-filter: blur(20px);
            border-radius: 22px; padding: 22px 18px;
            border: 1px solid {GLASS_BORDER}; text-align: center;
            box-shadow: 0 16px 34px -12px rgba(99, 102, 241, 0.22);
            transition: transform 220ms ease-out, box-shadow 220ms ease-out;
            animation: popIn 550ms cubic-bezier(0.22,1,0.36,1) both;
        }}
        .mm-metric:hover {{
            transform: translateY(-6px) scale(1.04) rotate(-1deg);
            box-shadow: 0 24px 44px -12px rgba(236, 72, 153, 0.4);
        }}
        .mm-metric .mm-label {{ color: {MUTED}; font-size: 11.5px; font-weight: 700; letter-spacing: 0.06em; text-transform: uppercase; }}
        .mm-metric .mm-value {{
            font-size: 30px; font-weight: 800; margin-top: 6px;
            background: linear-gradient(135deg, {PINK}, {PRIMARY}, {CYAN});
            background-size: 200% auto;
            -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text;
            animation: gradientShift 4s ease infinite;
        }}
        .mm-metric .mm-sub {{ font-size: 12px; color: {GREEN}; font-weight: 700; margin-top: 4px; }}

        /* ---- Badges — subtle pulse ---- */
        .mm-badge-positive {{
            display:inline-block; background: linear-gradient(135deg, {GREEN}, #059669);
            color: white; padding:5px 14px; border-radius:20px; font-size:12.5px; font-weight:800;
            box-shadow: 0 6px 14px -4px rgba(16,185,129,0.5);
            animation: badgePulse 2.2s ease-in-out infinite;
        }}
        @keyframes badgePulse {{
            0%, 100% {{ transform: scale(1); }}
            50% {{ transform: scale(1.06); }}
        }}

        /* ---- Header — animated moving gradient text ---- */
        .mm-header {{ display:flex; justify-content:space-between; align-items:center; padding-bottom: 10px; margin-bottom: 14px; }}
        .mm-header h2 {{
            margin: 0; font-weight: 800;
            background: linear-gradient(135deg, {PINK} 0%, {PRIMARY} 50%, {CYAN} 100%);
            background-size: 200% auto;
            -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text;
            animation: gradientShift 5s ease infinite;
        }}
        .mm-header p {{ margin: 3px 0 0 0; color:{MUTED}; font-size: 14px; font-weight: 500; }}

        /* ---- Buttons — gradient shift + shine sweep + bold hover ---- */
        div.stButton > button,
        .stFormSubmitButton > button,
        button[kind="primary"],
        button[kind="secondary"],
        div[data-testid="stButton"] button,
        div[data-testid="stFormSubmitButton"] button {{
            position: relative; overflow: hidden;
            border-radius: 16px !important;
            font-weight: 700 !important;
            border: none !important;
            transition: transform 200ms ease-out, box-shadow 200ms ease-out, background 200ms ease-out !important;
        }}
        div.stButton > button[kind="primary"],
        .stFormSubmitButton > button[kind="primary"],
        button[kind="primary"],
        div[data-testid="stButton"] button[kind="primary"],
        div[data-testid="stFormSubmitButton"] button[kind="primary"] {{
            background: {BTN_GRADIENT} !important;
            background-size: 200% 200% !important;
            color: white !important;
            box-shadow: 0 14px 28px -10px rgba(236, 72, 153, 0.5) !important;
            animation: gradientShift 3s ease infinite !important;
        }}
        div.stButton > button[kind="primary"]::after,
        .stFormSubmitButton > button[kind="primary"]::after,
        button[kind="primary"]::after {{
            content: ""; position: absolute; top: 0; left: -60%; width: 40%; height: 100%;
            background: linear-gradient(120deg, transparent, rgba(255,255,255,0.5), transparent);
            transform: skewX(-20deg);
            animation: shineSweep 2.8s ease-in-out infinite;
        }}
        @keyframes shineSweep {{
            0%   {{ left: -60%; }}
            50%  {{ left: 120%; }}
            100% {{ left: 120%; }}
        }}
        div.stButton > button[kind="primary"]:hover,
        .stFormSubmitButton > button[kind="primary"]:hover,
        button[kind="primary"]:hover,
        div[data-testid="stButton"] button[kind="primary"]:hover,
        div[data-testid="stFormSubmitButton"] button[kind="primary"]:hover {{
            background: {BTN_HOVER} !important;
            transform: translateY(-4px) scale(1.025);
            box-shadow: 0 22px 40px -10px rgba(99, 102, 241, 0.6) !important;
        }}
        div.stButton > button:not([kind="primary"]),
        div[data-testid="stButton"] button:not([kind="primary"]) {{
            background: linear-gradient(135deg, rgba(236,72,153,0.10), rgba(99,102,241,0.10)) !important;
            color: {TITLE_COLOR} !important;
            border: 1.5px solid rgba(99,102,241,0.18) !important;
        }}
        div.stButton > button:not([kind="primary"]):hover,
        div[data-testid="stButton"] button:not([kind="primary"]):hover {{
            background: linear-gradient(135deg, rgba(236,72,153,0.22), rgba(99,102,241,0.22)) !important;
            transform: translateY(-3px) scale(1.02);
        }}

        /* ---- Text inputs ---- */
        .stTextInput > div > div > input,
        div[data-testid="stTextInput"] input,
        .stTextArea textarea,
        div[data-testid="stTextArea"] textarea {{
            border-radius: 16px !important;
            border: 1.5px solid rgba(99,102,241,0.2) !important;
            background: rgba(255,255,255,0.85) !important;
            transition: border-color 200ms ease-out, box-shadow 200ms ease-out, transform 150ms ease-out !important;
        }}
        .stTextInput > div > div > input:focus,
        .stTextArea textarea:focus {{
            border-color: {PINK} !important;
            box-shadow: 0 0 0 5px rgba(236, 72, 153, 0.22) !important;
            transform: scale(1.005);
        }}

        /* ---- Journal: warm coral/amber, pulsing ---- */
        .journal-card {{
            position: relative;
            background: rgba(255, 255, 255, 0.9);
            border-radius: 26px; padding: 30px 32px;
            box-shadow: 0 20px 44px -16px rgba(249, 112, 102, 0.3);
            margin-bottom: 22px;
            border: 1.5px solid rgba(255,255,255,0.9);
            animation: popIn 550ms cubic-bezier(0.22,1,0.36,1);
        }}
        .journal-header {{ text-align: center; padding: 8px 0 22px 0; }}
        .journal-header .j-emoji {{ font-size: 2.6rem; line-height: 1; display:inline-block; animation: wiggle 2.4s ease-in-out infinite; }}
        @keyframes wiggle {{
            0%, 100% {{ transform: rotate(0deg); }}
            25%  {{ transform: rotate(-8deg) scale(1.05); }}
            75%  {{ transform: rotate(8deg) scale(1.05); }}
        }}
        .journal-header .j-title {{
            font-size: 1.25rem; font-weight: 800; margin-top: 8px;
            background: linear-gradient(135deg, {AMBER}, {CORAL});
            background-size: 200% auto;
            -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text;
            animation: gradientShift 4s ease infinite;
        }}
        .journal-card div.stButton > button[kind="primary"] {{
            background: linear-gradient(135deg, {AMBER} 0%, {CORAL} 100%) !important;
            background-size: 200% 200% !important;
            box-shadow: 0 14px 28px -10px rgba(249, 112, 102, 0.5) !important;
        }}
        .journal-card div.stButton > button[kind="primary"]:hover {{
            background: linear-gradient(135deg, {CORAL} 0%, #DC2626 100%) !important;
        }}
        .journal-card .stTextArea textarea {{
            border-radius: 18px !important;
            background: rgba(255,251,240,0.85) !important;
            border: 1.5px solid rgba(245,158,11,0.22) !important;
        }}
        .journal-card .streamlit-expanderHeader {{
            border-radius: 15px !important;
            background: rgba(255,247,230,0.7) !important;
        }}

        /* ---- Calendar: cyan/teal, glowing pulsing today, staggered day pop ---- */
        .calendar-card {{
            background: rgba(255,255,255,0.87);
            border-radius: 26px; padding: 0;
            box-shadow: 0 20px 44px -16px rgba(6, 182, 212, 0.3);
            margin-bottom: 22px; overflow: hidden;
            animation: popIn 550ms cubic-bezier(0.22,1,0.36,1);
        }}
        .calendar-header-bar {{
            background: linear-gradient(135deg, {CYAN} 0%, {PRIMARY} 100%);
            background-size: 200% 200%;
            animation: gradientShift 5s ease infinite;
            padding: 20px 28px; display: flex; justify-content: space-between; align-items: center;
        }}
        .calendar-header-bar .cal-title {{ font-size: 1.3rem; font-weight: 800; color: white; }}
        .calendar-body {{ padding: 22px 28px; }}
        .weekday-pill {{
            background: linear-gradient(135deg, rgba(6,182,212,0.15), rgba(99,102,241,0.15));
            color: {TITLE_COLOR};
            aspect-ratio: 1; border-radius: 50%;
            display: flex; align-items: center; justify-content: center;
            font-weight: 800; font-size: 0.85rem; margin: 0 auto; width: 34px; height: 34px;
        }}
        .day-pill {{
            aspect-ratio: 1; border-radius: 14px;
            display: flex; flex-direction: column; align-items: center; justify-content: center;
            border: 1.5px solid rgba(99,102,241,0.08);
            transition: transform 180ms ease-out, box-shadow 180ms ease-out;
            padding: 6px 2px;
            animation: popIn 450ms cubic-bezier(0.22,1,0.36,1) both;
        }}
        .day-pill:hover {{ transform: scale(1.15) rotate(-2deg); box-shadow: 0 10px 22px -6px rgba(99,102,241,0.45); z-index: 2; }}
        .day-pill.today {{
            border-color: {PINK}; font-weight: 800; color: {PINK}; border-width: 2.5px;
            animation: todayPulse 2s ease-in-out infinite, popIn 450ms cubic-bezier(0.22,1,0.36,1) both;
        }}
        @keyframes todayPulse {{
            0%, 100% {{ box-shadow: 0 0 0 3px rgba(236,72,153,0.18); }}
            50%      {{ box-shadow: 0 0 0 7px rgba(236,72,153,0.28); }}
        }}
        .day-num {{ font-size: 11px; font-weight: 700; }}
        .calendar-legend-bar {{
            background: linear-gradient(135deg, rgba(255,251,235,0.9), rgba(207,250,254,0.9));
            padding: 18px 28px; display: flex; justify-content: space-between; flex-wrap: wrap; gap: 10px;
            font-size: 0.8rem; font-weight: 700; color: {INK};
        }}

        /* ---- Login page — glowing, sparkling, floating blobs ---- */
        .login-shell {{
            position: relative; overflow: visible;
            background: {GLASS_BG};
            backdrop-filter: blur(24px);
            border-radius: 32px;
            box-shadow: 0 30px 60px -18px rgba(99,102,241,0.3);
            border: 1.5px solid {GLASS_BORDER};
            animation: popIn 650ms cubic-bezier(0.22,1,0.36,1), glowPulse 5s ease-in-out infinite 700ms;
        }}
        .login-blob-1, .login-blob-2 {{
            position: absolute; border-radius: 50%; filter: blur(50px); z-index: -1; pointer-events:none;
        }}
        .login-blob-1 {{
            width: 220px; height: 220px; background: radial-gradient(circle, rgba(236,72,153,0.35), transparent 70%);
            top: -60px; left: -60px; animation: blobFloat 9s ease-in-out infinite;
        }}
        .login-blob-2 {{
            width: 260px; height: 260px; background: radial-gradient(circle, rgba(6,182,212,0.3), transparent 70%);
            bottom: -80px; right: -60px; animation: blobFloat 11s ease-in-out infinite reverse;
        }}
        .brand-row {{
            font-size: 1.7rem; font-weight: 900;
            background: linear-gradient(135deg, {PINK}, {PRIMARY}, {CYAN});
            background-size: 200% auto;
            -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text;
            display:flex; align-items:center; gap:8px; margin-bottom: 4px;
            animation: gradientShift 4s ease infinite;
        }}
        .quote-box {{
            background: linear-gradient(90deg, rgba(236,72,153,0.08), rgba(99,102,241,0.08));
            border-left: 4px solid {PINK};
            padding: 14px 18px; border-radius: 0 14px 14px 0;
            font-style: italic; font-weight: 600; color: {INK}; margin: 18px 0 24px 0;
        }}
        .greet-title {{
            font-size: 1.7rem; font-weight: 800; margin-bottom: 3px;
            background: linear-gradient(135deg, {INK}, {PRIMARY}, {PINK});
            background-size: 200% auto;
            -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text;
            animation: gradientShift 5s ease infinite;
        }}
        .greet-sub {{ color: {MUTED}; font-size: 0.98rem; margin-bottom: 22px; font-weight: 500; }}
        .divider-text {{
            text-align:center; color:{MUTED}; font-size:0.85rem; margin: 20px 0;
            display:flex; align-items:center; gap:12px; font-weight: 600;
        }}
        .divider-text::before, .divider-text::after {{
            content:''; flex:1; border-bottom:1.5px solid rgba(99,102,241,0.15);
        }}
        .stars-twinkle {{ display:inline-block; animation: twinkle 1.6s ease-in-out infinite; }}
        @keyframes twinkle {{
            0%, 100% {{ opacity: 1; transform: scale(1); }}
            50%      {{ opacity: 0.55; transform: scale(0.92); }}
        }}
    </style>
    """, unsafe_allow_html=True)

def donut_chart(counts: dict, size=2.6):
    """Small donut chart themed to the mood/emotion colors — used for
    'Top Emotions' / 'Emotion Distribution' style widgets."""
    labels, values, colors = [], [], []
    for k, v in counts.items():
        if v > 0:
            labels.append(k); values.append(v)
            colors.append(style_for(k)["color"])
    if not values:
        return None
    fig, ax = plt.subplots(figsize=(size, size))
    ax.pie(values, colors=colors, startangle=90, wedgeprops=dict(width=0.38, edgecolor="white"))
    ax.set(aspect="equal")
    fig.patch.set_alpha(0.0)
    return fig

def metric_tile(label, value, sub=None, delay=0):
    sub_html = f"<div class='mm-sub'>{sub}</div>" if sub else ""
    st.markdown(
        f"<div class='mm-metric' style='animation-delay:{delay}ms'><div class='mm-label'>{label}</div>"
        f"<div class='mm-value'>{value}</div>{sub_html}</div>",
        unsafe_allow_html=True,
    )

inject_css()

@st.cache_resource
def setup(): init_db()
setup()

if "page" not in st.session_state: st.session_state.page = "welcome"
if "auth_mode" not in st.session_state: st.session_state.auth_mode = "login"
if "token" not in st.session_state: st.session_state.token = None
if "email" not in st.session_state: st.session_state.email = None
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "cal_year" not in st.session_state: st.session_state.cal_year = date.today().year
if "cal_month" not in st.session_state: st.session_state.cal_month = date.today().month
if "today_mood_saved" not in st.session_state: st.session_state.today_mood_saved = False
if "nav" not in st.session_state: st.session_state.nav = "Home"

def goto_auth(mode): st.session_state.auth_mode = mode; st.rerun()

def valid_pw(pw):
    return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)


if st.session_state.token:
    user = read_token(st.session_state.token)

    if user:
        role = user.get("role", "employee")
        headers = {"Authorization": f"Bearer {st.session_state.token}"}

        with st.sidebar:
            st.markdown(
                f"<div style='display:flex;align-items:center;gap:8px;padding:6px 4px 18px 4px'>"
                f"<span style='font-size:22px'>🧠</span>"
                f"<span style='font-size:18px;font-weight:800;color:{INK}'>Mood<span style='color:{BRAND_GREEN}'>Mentor</span></span>"
                f"</div>",
                unsafe_allow_html=True,
            )

            if role == "employee":
                nav_options = [
                    "Home",
                    "Analyze Text",
                    "Journal",
                    "Wellness Chat",
                    "Face Recognition",
                    "Dashboard",
                    "Weekly Report"
                ]
            else:
                nav_options = ["Reports"]

            st.session_state.nav = st.radio(
                "Navigate",
                nav_options,
                index=nav_options.index(st.session_state.nav)
                if st.session_state.nav in nav_options else 0,
                label_visibility="collapsed",
            )

            st.divider()
            st.caption(f"Signed in as **{user['username']}**")
            st.caption(f"{user['email']} · {role.capitalize()}")

            if st.button("Log out", use_container_width=True):
                st.session_state.token = None
                st.session_state.page = "welcome"
                st.rerun()

        greeting = (
            "Good Morning"
            if datetime.now().hour < 12
            else (
                "Good Afternoon"
                if datetime.now().hour < 18
                else "Good Evening"
            )
        )

        st.markdown(
            f"<div class='mm-header'><div><h2>{greeting}, {user['username']}! 👋</h2>"
            f"<p>Here's your emotional wellness overview.</p></div></div>",
            unsafe_allow_html=True,
        )

        if role == "employee":
            section = st.session_state.nav

            if section == "Home":
                history_all = get_user_mood_history(user["id"], limit=500)
                latest = history_all[0] if history_all else None
                today_count = sum(
                    1 for h in history_all
                    if h["mood_date"] == date.today()
                )

                streak = 0
                day_ptr = date.today()
                day_set = {h["mood_date"] for h in history_all}

                while day_ptr in day_set:
                    streak += 1
                    day_ptr = date.fromordinal(
                        day_ptr.toordinal() - 1
                    )

                positive_count = sum(
                    1 for h in history_all
                    if h["sentiment"] in ("Amazing", "Happy")
                )

                overall_score = (
                    int(100 * positive_count / len(history_all))
                    if history_all else 0
                )

                m1, m2, m3, m4 = st.columns(4)

                with m1:
                    if latest:
                        s = style_for(latest["sentiment"])
                        metric_tile(
                            "Current Mood",
                            f"{s['emoji']} {latest['sentiment']}",
                            delay=0
                        )
                    else:
                        metric_tile(
                            "Current Mood",
                            "—",
                            delay=0
                        )

                with m2:
                    metric_tile(
                        "Overall Score",
                        f"{overall_score}%",
                        "Positive" if overall_score >= 50 else "Needs care",
                        delay=80
                    )

                with m3:
                    metric_tile(
                        "Entries Today",
                        today_count,
                        delay=160
                    )

                with m4:
                    metric_tile(
                        "Current Streak",
                        f"{streak} Days",
                        delay=240
                    )

                st.write("")

                st.markdown(
                    "<div class='mm-card'>",
                    unsafe_allow_html=True
                )

                st.subheader("How Do You Feel?")

                now = datetime.now()

                st.caption(
                    f"📅 {now.strftime('%Y-%m-%d')}  "
                    f"🕒 {now.strftime('%H:%M')}"
                )

                cols = st.columns(len(MOOD_LABELS))
                picked = st.session_state.get("picked_mood")

                for col, label in zip(cols, MOOD_LABELS):
                    s = style_for(label)

                    with col:
                        st.markdown(
                            f"<div style='text-align:center;font-size:40px;"
                            f"filter:drop-shadow(0 6px 10px {s['glow']});"
                            f"animation:wiggle 3s ease-in-out infinite;"
                            f"display:block'>{s['emoji']}</div>"
                            f"<div style='text-align:center;"
                            f"color:{s['color']};font-weight:700'>{label}</div>",
                            unsafe_allow_html=True,
                        )

                        if st.button(
                            "Select",
                            key=f"pick_{label}",
                            use_container_width=True
                        ):
                            st.session_state.picked_mood = label

                st.write("")

                confirm_col = st.columns([3, 1, 3])[1]

                with confirm_col:
                    disabled = picked is None

                    if st.button(
                        "Save mood",
                        type="primary",
                        disabled=disabled,
                        use_container_width=True
                    ):
                        save_manual_mood(
                            user["id"],
                            st.session_state.picked_mood
                        )

                        st.session_state.today_mood_saved = True
                        st.session_state.picked_mood = None
                        st.rerun()

                if st.session_state.today_mood_saved:
                    st.success("Today's mood saved!")
                    st.session_state.today_mood_saved = False

                st.markdown(
                    "</div>",
                    unsafe_allow_html=True
                )

                # ---- Calendar ----

                st.markdown(
                    '<div class="calendar-card">',
                    unsafe_allow_html=True
                )

                st.markdown(
                    '<div class="calendar-header-bar">',
                    unsafe_allow_html=True
                )

                st.markdown(
                    f'<div class="cal-title">'
                    f'{calendar.month_name[st.session_state.cal_month]} '
                    f'{st.session_state.cal_year}</div>',
                    unsafe_allow_html=True,
                )

                st.markdown(
                    '</div>',
                    unsafe_allow_html=True
                )

                st.markdown(
                    '<div class="calendar-body">',
                    unsafe_allow_html=True
                )

                nav_l, nav_mid, nav_r = st.columns([1, 4, 1])

                if nav_l.button("‹ Prev"):
                    m = st.session_state.cal_month - 1
                    y = st.session_state.cal_year

                    if m == 0:
                        m, y = 12, y - 1

                    st.session_state.cal_month = m
                    st.session_state.cal_year = y
                    st.rerun()

                if nav_r.button("Next ›"):
                    m = st.session_state.cal_month + 1
                    y = st.session_state.cal_year

                    if m == 13:
                        m, y = 1, y + 1

                    st.session_state.cal_month = m
                    st.session_state.cal_year = y
                    st.rerun()

                logs = get_mood_logs_for_month(
                    user["id"],
                    st.session_state.cal_year,
                    st.session_state.cal_month
                )

                by_day = {
                    row["mood_date"].day: row
                    for row in logs
                }

                weeks = calendar.Calendar(
                    firstweekday=6
                ).monthdayscalendar(
                    st.session_state.cal_year,
                    st.session_state.cal_month
                )

                day_names = [
                    "Su", "Mo", "Tu", "We",
                    "Th", "Fr", "Sa"
                ]

                header_cols = st.columns(7)

                for c, name in zip(header_cols, day_names):
                    c.markdown(
                        f"<div class='weekday-pill'>{name}</div>",
                        unsafe_allow_html=True
                    )

                today = date.today()
                day_counter = 0

                for week in weeks:
                    cols = st.columns(7)

                    for col, day_num in zip(cols, week):

                        if day_num == 0:
                            col.write("")
                            continue

                        entry = by_day.get(day_num)

                        s = style_for(
                            entry["sentiment"] if entry else None
                        )

                        is_today = (
                            day_num == today.day
                            and st.session_state.cal_month == today.month
                            and st.session_state.cal_year == today.year
                        )

                        today_class = " today" if is_today else ""

                        border_style = (
                            f"border-color:{s['border']};"
                            if entry and not is_today
                            else ""
                        )

                        bg_style = (
                            f"background:{s['bg']};"
                            if entry
                            else "background:#FFFFFF;"
                        )

                        emoji_html = (
                            f"<div style='font-size:1.1rem;"
                            f"line-height:1'>{s['emoji']}</div>"
                            if entry
                            else ""
                        )

                        delay_ms = (day_counter % 14) * 25
                        day_counter += 1

                        col.markdown(
                            f"<div class='day-pill{today_class}' "
                            f"style='{bg_style}{border_style}"
                            f"animation-delay:{delay_ms}ms'>"
                            f"<div class='day-num'>{day_num}</div>"
                            f"{emoji_html}</div>",
                            unsafe_allow_html=True,
                        )

                st.markdown(
                    '</div>',
                    unsafe_allow_html=True
                )

                legend_html = "".join(
                    f"<span>{style_for(l)['emoji']} {l}</span>"
                    for l in MOOD_LABELS
                )

                st.markdown(
                    f'<div class="calendar-legend-bar">'
                    f'{legend_html}</div>',
                    unsafe_allow_html=True,
                )

                st.markdown(
                    '</div>',
                    unsafe_allow_html=True
                )

                # ---- Daily Wellness Details ----

                st.markdown(
                    "<div class='mm-card'>",
                    unsafe_allow_html=True
                )

                st.subheader("🌿 Daily Wellness Details")

                st.caption(
                    "Store stress, sleep and workload separately "
                    "from journal entries. Missing values remain unavailable."
                )

                wc1, wc2, wc3 = st.columns(3)

                with wc1:
                    stress_value = st.number_input(
                        "Stress level (0–10)",
                        min_value=0.0,
                        max_value=10.0,
                        value=5.0,
                        step=0.5,
                        key="daily_stress"
                    )

                with wc2:
                    sleep_value = st.number_input(
                        "Sleep hours",
                        min_value=0.0,
                        max_value=24.0,
                        value=7.0,
                        step=0.5,
                        key="daily_sleep"
                    )

                with wc3:
                    workload_value = st.selectbox(
                        "Workload",
                        ["Not recorded", "Low", "Medium", "High"],
                        key="daily_workload"
                    )

                missing_stress = st.checkbox(
                    "Stress not recorded",
                    key="missing_stress"
                )

                missing_sleep = st.checkbox(
                    "Sleep not recorded",
                    key="missing_sleep"
                )

                missing_workload = st.checkbox(
                    "Workload not recorded",
                    key="missing_workload"
                )

                if st.button(
                    "Save today's wellness details",
                    type="primary",
                    use_container_width=True
                ):
                    save_daily_wellness(
                        user["id"],
                        date.today(),
                        stress_level=None
                        if missing_stress
                        else stress_value,
                        sleep_hours=None
                        if missing_sleep
                        else sleep_value,
                        workload=None
                        if missing_workload
                        or workload_value == "Not recorded"
                        else workload_value,
                    )

                    st.success(
                        "Today's wellness details saved. "
                        "No journal entry was created."
                    )

                    st.rerun()

                st.markdown(
                    "</div>",
                    unsafe_allow_html=True
                )

            elif section == "Analyze Text":

                st.markdown(
                    "<div class='mm-card'>",
                    unsafe_allow_html=True
                )

                st.subheader("📝 Analyze Text")

                st.caption(
                    "Enter your text below and let AI analyze your emotions."
                )

                text_in = st.text_area(
                    "Type or paste your text here…",
                    height=160,
                    label_visibility="collapsed",
                    placeholder="Type or paste your text here…"
                )

                st.caption(
                    f"{len(text_in)}/5000 characters"
                )

                if st.button(
                    "Analyze Now",
                    type="primary",
                    use_container_width=True
                ):

                    if not text_in.strip():
                        st.warning("Write something first.")

                    else:

                        with st.spinner("Running NLP analysis…"):

                            try:
                                resp = requests.post(
                                    f"{BACKEND_URL}/analyze-text",
                                    json={"text": text_in},
                                    headers=headers,
                                    timeout=120,
                                )

                            except requests.exceptions.RequestException as e:
                                st.error(
                                    f"Could not reach backend: {e}"
                                )
                                resp = None

                        if resp is not None:

                            if resp.status_code != 200:
                                st.error("Analysis failed.")

                            else:

                                r = resp.json()

                                confidence = r.get(
                                    "emotion_confidence"
                                )

                                save_mood_log(
                                    user["id"],
                                    r["final_sentiment"],
                                    r["final_emotion"],
                                    r["sentiment_scores"]["compound"],
                                    text_in,
                                    confidence=confidence,
                                    positive_score=r["sentiment_scores"].get("pos"),
                                    negative_score=r["sentiment_scores"].get("neg"),
                                    neutral_score=r["sentiment_scores"].get("neu"),
                                    detected_language=r.get("detected_language"),
                                    cleaned_text=r.get("cleaned_text"),
                                )

                                st.subheader("Analysis Results")

                                rc1, rc2 = st.columns(2)

                                with rc1:
                                    st.write("**Overall Emotion**")

                                    s = style_for(
                                        r["final_sentiment"]
                                    )

                                    conf_label = (
                                        f"Confidence: {confidence:.0%}"
                                        if confidence is not None
                                        else ""
                                    )

                                    st.markdown(
                                        f"### {s['emoji']} "
                                        f"{r['final_emotion']}"
                                        + (
                                            f"&nbsp;&nbsp;"
                                            f"<span style='font-size:14px;"
                                            f"color:#6b7280;"
                                            f"font-weight:600'>"
                                            f"{conf_label}</span>"
                                            if conf_label
                                            else ""
                                        ),
                                        unsafe_allow_html=True,
                                    )

                                    badge = "mm-badge-positive"

                                    st.markdown(
                                        f"<span class='{badge}'>"
                                        f"{r['final_sentiment']}</span>"
                                        f"&nbsp;&nbsp;Score: "
                                        f"**{r['sentiment_scores']['compound']:.2f}**",
                                        unsafe_allow_html=True,
                                    )

                                with rc2:
                                    st.write(
                                        "**Emotion Distribution**"
                                    )

                                    fig = donut_chart(
                                        r["emotion_scores"]
                                    )

                                    if fig:
                                        st.pyplot(
                                            fig,
                                            use_container_width=False
                                        )
                                    else:
                                        st.bar_chart(
                                            r["emotion_scores"]
                                        )

                st.markdown(
                    "</div>",
                    unsafe_allow_html=True
                )

            elif section == "Journal":

                st.markdown(
                    '<div class="journal-card">',
                    unsafe_allow_html=True
                )

                st.markdown(
                    '<div class="journal-header">'
                    '<div class="j-emoji">😍</div>'
                    '<div class="j-title">'
                    'Welcome Back! Let\'s Journal Your Day'
                    '</div>'
                    '</div>',
                    unsafe_allow_html=True,
                )

                journal_text = st.text_area(
                    "Write about how you're feeling today",
                    height=150,
                    placeholder="What's making you smile today?",
                )

                if st.button("Analyze my entry"):

                    if not journal_text.strip():
                        st.warning("Write something first.")

                    else:

                        with st.spinner(
                            "Running NLP analysis…"
                        ):

                            try:
                                resp = requests.post(
                                    f"{BACKEND_URL}/analyze-text",
                                    json={"text": journal_text},
                                    headers=headers,
                                    timeout=120,
                                )

                            except requests.exceptions.RequestException as e:
                                st.error(
                                    f"Could not reach backend: {e}"
                                )
                                resp = None

                        if resp is not None:

                            if resp.status_code != 200:
                                st.error("Analysis failed.")

                            else:

                                r = resp.json()

                                confidence = r.get(
                                    "emotion_confidence"
                                )

                                save_mood_log(
                                    user["id"],
                                    r["final_sentiment"],
                                    r["final_emotion"],
                                    r["sentiment_scores"]["compound"],
                                    journal_text,
                                    confidence=confidence,
                                    positive_score=r["sentiment_scores"].get("pos"),
                                    negative_score=r["sentiment_scores"].get("neg"),
                                    neutral_score=r["sentiment_scores"].get("neu"),
                                    detected_language=r.get("detected_language"),
                                    cleaned_text=r.get("cleaned_text"),
                                )

                                conf_str = (
                                    f", Confidence: **{confidence:.0%}**"
                                    if confidence is not None
                                    else ""
                                )

                                st.success(
                                    f"Saved! Sentiment: "
                                    f"**{r['final_sentiment']}**, "
                                    f"Emotion: **{r['final_emotion']}**"
                                    f"{conf_str}"
                                )

                                st.bar_chart(
                                    r["emotion_scores"]
                                )

                st.markdown(
                    "</div>",
                    unsafe_allow_html=True
                )

                st.markdown(
                    '<div class="journal-card">',
                    unsafe_allow_html=True
                )

                st.subheader("📁 Or upload a file")

                uploaded = st.file_uploader(
                    "Choose a CSV or TXT file",
                    type=["csv", "txt"]
                )

                if uploaded is not None and st.button(
                    "Run NLP Analysis on file"
                ):

                    files = {
                        "file": (
                            uploaded.name,
                            uploaded.getvalue()
                        )
                    }

                    with st.spinner(
                        "Running multilingual NLP pipeline…"
                    ):

                        try:
                            resp = requests.post(
                                f"{BACKEND_URL}/analyze",
                                files=files,
                                headers=headers,
                                timeout=120
                            )

                        except requests.exceptions.RequestException as e:
                            st.error(
                                f"Could not reach backend: {e}"
                            )
                            resp = None

                    if resp is not None:

                        if resp.status_code != 200:
                            st.error("Analysis failed.")

                        else:

                            r = resp.json()

                            confidence = r.get(
                                "emotion_confidence"
                            )

                            save_mood_log(
                                user["id"],
                                r["final_sentiment"],
                                r["final_emotion"],
                                r["sentiment_scores"]["compound"],
                                r.get("cleaned_text", ""),
                                confidence=confidence,
                                positive_score=r["sentiment_scores"].get("pos"),
                                negative_score=r["sentiment_scores"].get("neg"),
                                neutral_score=r["sentiment_scores"].get("neu"),
                                detected_language=r.get("detected_language"),
                                cleaned_text=r.get("cleaned_text"),
                            )

                            conf_str = (
                                f", Confidence: **{confidence:.0%}**"
                                if confidence is not None
                                else ""
                            )

                            st.success(
                                f"Saved! Sentiment: "
                                f"**{r['final_sentiment']}**, "
                                f"Emotion: **{r['final_emotion']}**"
                                f"{conf_str}"
                            )

                            st.bar_chart(
                                r["emotion_scores"]
                            )

                st.markdown(
                    "</div>",
                    unsafe_allow_html=True
                )

                st.markdown(
                    '<div class="journal-card">',
                    unsafe_allow_html=True
                )

                st.subheader("📜 Past entries")

                history = [
                    h
                    for h in get_user_mood_history(
                        user["id"],
                        limit=20
                    )
                    if h["journal_text"]
                ]

                if not history:
                    st.caption(
                        "No journal entries yet."
                    )

                for h in history:

                    s = style_for(
                        h["sentiment"]
                    )

                    conf_str = (
                        f" · Confidence: "
                        f"{h['confidence']:.0%}"
                        if h.get("confidence") is not None
                        else ""
                    )

                    with st.expander(
                        f"{s['emoji']} "
                        f"{h['sentiment']} — "
                        f"{h['created_at'].strftime('%Y-%m-%d %H:%M')}"
                        f"{conf_str}"
                    ):
                        st.write(
                            h["journal_text"]
                        )

                st.markdown(
                    "</div>",
                    unsafe_allow_html=True
                )

            elif section == "Face Recognition":

                st.markdown(
                    "<div class='mm-card'>",
                    unsafe_allow_html=True
                )

                st.subheader("😊 Face Recognition")

                st.caption(
                    "Allow camera access to capture your face."
                )

                camera_image = st.camera_input(
                    "Take a photo"
                )

                if camera_image is not None:

                    st.image(
                        camera_image,
                        caption="Captured image",
                        use_container_width=True
                    )

                    st.success(
                        "Face image captured successfully!"
                    )

                    st.write("")

                    analyze_face = st.button(
                        "🔍 Analyze Face",
                        type="primary",
                        use_container_width=True
                    )

                    if analyze_face:

                        with st.spinner(
                            "Analyzing your face..."
                        ):

                            try:

                                import tempfile
                                from deepface import DeepFace

                                image_bytes = camera_image.getvalue()

                                with tempfile.NamedTemporaryFile(
                                    delete=False,
                                    suffix=".jpg"
                                ) as tmp_file:

                                    tmp_file.write(
                                        image_bytes
                                    )

                                    image_path = tmp_file.name

                                result = DeepFace.analyze(
                                    img_path=image_path,
                                    actions=[
                                        "emotion"
                                    ],
                                    enforce_detection=False
                                )

                                if isinstance(result, list):
                                    result = result[0]

                                emotion = result.get(
                                    "dominant_emotion",
                                    "Unknown"
                                )

                                emotion_scores = result.get(
                                    "emotion",
                                    {}
                                )

                                confidence = emotion_scores.get(
                                    emotion,
                                    0
                                )

                                st.success(
                                    "✅ Face analyzed successfully!"
                                )

                                st.markdown(
                                    "### 📊 Face Analysis Result"
                                )

                                col1, col2 = st.columns(2)

                                with col1:

                                    st.metric(
                                        "Detected Emotion",
                                        str(emotion).capitalize()
                                    )

                                with col2:

                                    st.metric(
                                        "Confidence",
                                        f"{float(confidence):.1f}%"
                                    )

                                st.progress(
                                    min(
                                        max(
                                            float(confidence) / 100,
                                            0.0
                                        ),
                                        1.0
                                    )
                                )

                                st.caption(
                                    "The percentage represents the model's "
                                    "confidence for the detected emotion."
                                )

                            except Exception as e:

                                st.error(
                                    "Unable to analyze the face."
                                )

                                st.code(
                                    str(e)
                                )

                st.markdown(
                    "</div>",
                    unsafe_allow_html=True
                )

            elif section == "Wellness Chat":

                st.markdown(
                    "<div class='mm-card'>",
                    unsafe_allow_html=True
                )

                st.subheader("💬 Wellness Chat")

                st.caption(
                    "A supportive space to talk about how you're feeling. "
                    "Not a substitute for professional care."
                )

                chat_box = st.container(
                    height=450
                )

                with chat_box:

                    for turn in st.session_state.chat_history:

                        with st.chat_message(
                            turn["role"]
                        ):

                            st.write(
                                turn["content"]
                            )

                user_msg = st.chat_input(
                    "How are you feeling today?"
                )

                if user_msg:

                    st.session_state.chat_history.append(
                        {
                            "role": "user",
                            "content": user_msg
                        }
                    )

                    recent_history = (
                        st.session_state.chat_history[-10:-1]
                    )

                    try:

                        resp = requests.post(
                            f"{BACKEND_URL}/chat",
                            json={
                                "message": user_msg,
                                "history": recent_history
                            },
                            headers=headers,
                            timeout=60
                        )

                        reply = (
                            resp.json()["reply"]
                            if resp.status_code == 200
                            else
                            "Sorry, I couldn't reach the wellness assistant right now."
                        )

                    except requests.exceptions.RequestException:

                        reply = (
                            "Sorry, I couldn't reach the wellness assistant right now."
                        )

                    st.session_state.chat_history.append(
                        {
                            "role": "assistant",
                            "content": reply
                        }
                    )

                    st.rerun()

                if (
                    st.session_state.chat_history
                    and st.button("Clear chat")
                ):

                    st.session_state.chat_history = []

                    st.rerun()

                st.markdown(
                    "</div>",
                    unsafe_allow_html=True
                )

            elif section == "Weekly Report":

                st.markdown(
                    "<div class='mm-card'>",
                    unsafe_allow_html=True
                )

                st.subheader(
                    "📊 Weekly Wellness Report"
                )

                st.caption(
                    "A holistic 7-day assessment using actual stored mood, "
                    "journal/NLP, stress, sleep, workload and journal "
                    "consistency data. Missing values are never treated as zero."
                )

                end_date = st.date_input(
                    "Report end date",
                    value=date.today(),
                    key="weekly_end_date"
                )

                start_date = (
                    end_date
                    - __import__('datetime').timedelta(days=6)
                )

                st.caption(
                    f"Selected period: "
                    f"{start_date} → {end_date}"
                )

                with st.expander(
                    "⚙️ Configure scoring weights",
                    expanded=False
                ):

                    weight_values = {}
                    weight_cols = st.columns(2)

                    for idx, (
                        name,
                        default
                    ) in enumerate(
                        DEFAULT_WEEKLY_WEIGHTS.items()
                    ):

                        with weight_cols[idx % 2]:

                            weight_values[name] = st.slider(
                                name,
                                0,
                                50,
                                int(default),
                                1,
                                key=f"weekly_weight_{name}"
                            )

                    st.caption(
                        "Weights are normalized automatically across "
                        "the components that actually have data. "
                        "A missing component is not scored as zero."
                    )

                report = build_weekly_report_data(
                    user["id"],
                    end_date,
                    get_user_mood_history,
                    get_daily_wellness_range
                )

                stats = aggregate_week(
                    report["days"],
                    weight_values
                )

                if stats["coverage_days"] == 0:

                    st.info(
                        "No wellness information is stored for this "
                        "7-day period yet. Add a mood, journal entry, "
                        "or daily wellness details to generate the report."
                    )

                else:

                    score = stats["weekly_score"]

                    status = (
                        "Excellent"
                        if score is not None and score >= 85
                        else
                        "Good"
                        if score is not None and score >= 70
                        else
                        "Needs Attention"
                        if score is not None
                        else
                        "Unavailable"
                    )

                    m1, m2, m3, m4 = st.columns(4)

                    with m1:
                        metric_tile(
                            "Wellness Score",
                            f"{score:.0f} / 100"
                            if score is not None
                            else "—",
                            status
                        )

                    with m2:
                        metric_tile(
                            "Data Coverage",
                            f"{stats['coverage_days']} / 7",
                            f"{stats['coverage_pct']:.2f}%"
                        )

                    with m3:
                        metric_tile(
                            "Average Stress",
                            f"{stats['avg_stress']:.1f} / 10"
                            if stats.get("avg_stress") is not None
                            else "—"
                        )

                    with m4:
                        metric_tile(
                            "Average Sleep",
                            f"{stats['avg_sleep']:.1f} hrs"
                            if stats.get("avg_sleep") is not None
                            else "—"
                        )

                    m5, m6, m7, m8 = st.columns(4)

                    with m5:
                        metric_tile(
                            "Most Common Mood",
                            stats.get("most_common_mood") or "—"
                        )

                    with m6:
                        metric_tile(
                            "Most Common Emotion",
                            stats.get("most_common_emotion") or "—"
                        )

                    with m7:
                        metric_tile(
                            "Journal Activity",
                            f"{stats['journal_days']} / 7",
                            f"{stats['journal_consistency']:.2f}%"
                        )

                    with m8:
                        metric_tile(
                            "Emotion Confidence",
                            f"{stats['avg_emotion_confidence']:.1%}"
                            if stats.get("avg_emotion_confidence") is not None
                            else "—"
                        )

                    st.markdown(
                        "### 📅 Daily Wellness Scores"
                    )

                    table_rows = []

                    for d in report["days"]:

                        table_rows.append(
                            {
                                "Date": str(d["date"]),
                                "Mood": d.get("mood") or "—",
                                "Emotion": d.get("emotion") or "—",
                                "Stress":
                                    f"{d['stress_level']:.1f}/10"
                                    if d.get("stress_level") is not None
                                    else "—",
                                "Sleep":
                                    f"{d['sleep_hours']:.1f} h"
                                    if d.get("sleep_hours") is not None
                                    else "—",
                                "Workload":
                                    d.get("workload") or "—",
                                "Journal":
                                    "Yes"
                                    if d.get("has_journal")
                                    else "No",
                                "Daily Score":
                                    f"{d['daily_score']:.1f}"
                                    if d.get("daily_score") is not None
                                    else "—",
                            }
                        )

                    st.dataframe(
                        table_rows,
                        use_container_width=True,
                        hide_index=True
                    )

                    st.markdown(
                        "### 📈 Weekly Charts"
                    )

                    dates = [
                        d["date"].strftime("%a %d")
                        for d in report["days"]
                    ]

                    mood_numeric = [
                        MOOD_TO_NUM.get(
                            d.get("mood"),
                            None
                        )
                        for d in report["days"]
                    ]

                    stress_series = [
                        d.get("stress_level")
                        for d in report["days"]
                    ]

                    sleep_series = [
                        d.get("sleep_hours")
                        for d in report["days"]
                    ]

                    score_series = [
                        d.get("daily_score")
                        for d in report["days"]
                    ]

                    c1, c2 = st.columns(2)
                    figures = []

                    with c1:

                        fig1, ax1 = plt.subplots(
                            figsize=(6, 3.2)
                        )

                        ax1.plot(
                            dates,
                            [
                                v
                                if v is not None
                                else float('nan')
                                for v in mood_numeric
                            ],
                            marker="o"
                        )

                        ax1.set_title(
                            "Mood Trend"
                        )

                        ax1.set_ylabel(
                            "Mood score"
                        )

                        ax1.grid(
                            alpha=0.2
                        )

                        fig1.tight_layout()

                        st.pyplot(
                            fig1,
                            use_container_width=True
                        )

                        figures.append(
                            ("Mood Trend", fig1)
                        )

                    with c2:

                        fig2, ax2 = plt.subplots(
                            figsize=(6, 3.2)
                        )

                        ax2.plot(
                            dates,
                            [
                                v
                                if v is not None
                                else float('nan')
                                for v in stress_series
                            ],
                            marker="o"
                        )

                        ax2.set_title(
                            "Stress Trend"
                        )

                        ax2.set_ylabel(
                            "Stress / 10"
                        )

                        ax2.grid(
                            alpha=0.2
                        )

                        fig2.tight_layout()

                        st.pyplot(
                            fig2,
                            use_container_width=True
                        )

                        figures.append(
                            ("Stress Trend", fig2)
                        )

                    c3, c4 = st.columns(2)

                    with c3:

                        fig3, ax3 = plt.subplots(
                            figsize=(6, 3.2)
                        )

                        ax3.plot(
                            dates,
                            [
                                v
                                if v is not None
                                else float('nan')
                                for v in sleep_series
                            ],
                            marker="o"
                        )

                        ax3.set_title(
                            "Sleep Trend"
                        )

                        ax3.set_ylabel(
                            "Hours"
                        )

                        ax3.grid(
                            alpha=0.2
                        )

                        fig3.tight_layout()

                        st.pyplot(
                            fig3,
                            use_container_width=True
                        )

                        figures.append(
                            ("Sleep Trend", fig3)
                        )

                    with c4:

                        fig4, ax4 = plt.subplots(
                            figsize=(6, 3.2)
                        )

                        emo_counts = (
                            stats.get("emotion_counts")
                            or {}
                        )

                        if emo_counts:
                            ax4.pie(
                                list(emo_counts.values()),
                                labels=list(emo_counts.keys()),
                                autopct="%1.0f%%"
                            )
                        else:
                            ax4.text(
                                0.5,
                                0.5,
                                "No emotion data",
                                ha="center",
                                va="center"
                            )

                        ax4.set_title(
                            "Emotion Distribution"
                        )

                        fig4.tight_layout()

                        st.pyplot(
                            fig4,
                            use_container_width=True
                        )

                        figures.append(
                            (
                                "Emotion Distribution",
                                fig4
                            )
                        )

                    c5, c6 = st.columns(2)

                    with c5:

                        fig5, ax5 = plt.subplots(
                            figsize=(6, 3.2)
                        )

                        emo_counts = (
                            stats.get("emotion_counts")
                            or {}
                        )

                        if emo_counts:
                            ax5.bar(
                                list(emo_counts.keys()),
                                list(emo_counts.values())
                            )
                        else:
                            ax5.text(
                                0.5,
                                0.5,
                                "No emotion data",
                                ha="center",
                                va="center"
                            )

                        ax5.set_title(
                            "Emotion Frequency"
                        )

                        ax5.tick_params(
                            axis="x",
                            rotation=30
                        )

                        fig5.tight_layout()

                        st.pyplot(
                            fig5,
                            use_container_width=True
                        )

                        figures.append(
                            (
                                "Emotion Frequency",
                                fig5
                            )
                        )

                    with c6:

                        labels = [
                            "Positive",
                            "Negative",
                            "Neutral",
                            "Compound"
                        ]

                        vals = [
                            stats.get("avg_positive"),
                            stats.get("avg_negative"),
                            stats.get("avg_neutral"),
                            stats.get("avg_compound")
                        ]

                        if all(
                            v is None
                            for v in vals
                        ):
                            vals = [
                                0,
                                0,
                                0,
                                0
                            ]
                        else:
                            vals = [
                                0 if v is None else v
                                for v in vals
                            ]

                        fig6, ax6 = plt.subplots(
                            figsize=(6, 3.2)
                        )

                        ax6.bar(
                            labels,
                            vals
                        )

                        ax6.set_title(
                            "Sentiment Analysis"
                        )

                        ax6.grid(
                            axis="y",
                            alpha=0.2
                        )

                        fig6.tight_layout()

                        st.pyplot(
                            fig6,
                            use_container_width=True
                        )

                        figures.append(
                            (
                                "Sentiment Analysis",
                                fig6
                            )
                        )

                    fig7, ax7 = plt.subplots(
                        figsize=(12, 3.2)
                    )

                    ax7.plot(
                        dates,
                        [
                            v
                            if v is not None
                            else float('nan')
                            for v in score_series
                        ],
                        marker="o"
                    )

                    ax7.set_title(
                        "Wellness Score Trend"
                    )

                    ax7.set_ylabel(
                        "Score / 100"
                    )

                    ax7.grid(
                        alpha=0.2
                    )

                    fig7.tight_layout()

                    st.pyplot(
                        fig7,
                        use_container_width=True
                    )

                    figures.append(
                        (
                            "Wellness Score Trend",
                            fig7
                        )
                    )

                    st.markdown(
                        "### 🔎 Detailed Analysis"
                    )

                    a1, a2, a3 = st.columns(3)

                    with a1:

                        st.metric(
                            "Avg Stress",
                            f"{stats['avg_stress']:.1f}/10"
                            if stats.get("avg_stress") is not None
                            else "Unavailable"
                        )

                        if stats.get("stress_trend"):
                            st.caption(
                                f"Trend: {stats['stress_trend']}"
                            )

                        if (
                            stats.get("avg_stress") is not None
                            and stats["avg_stress"] >= 7
                        ):
                            st.warning(
                                "⚠ High average stress"
                            )

                    with a2:

                        st.metric(
                            "Avg Sleep",
                            f"{stats['avg_sleep']:.1f} hrs"
                            if stats.get("avg_sleep") is not None
                            else "Unavailable"
                        )

                        if (
                            stats.get("avg_sleep") is not None
                            and stats["avg_sleep"] < 6
                        ):
                            st.warning(
                                "⚠ Low average sleep"
                            )

                        if stats.get(
                            "sleep_consistency"
                        ) is not None:
                            st.caption(
                                f"Sleep consistency: "
                                f"{stats['sleep_consistency']:.1f}%"
                            )

                    with a3:

                        st.metric(
                            "High Workload Days",
                            stats.get(
                                "high_workload_days",
                                0
                            )
                        )

                        st.caption(
                            str(
                                stats.get(
                                    "workload_counts"
                                )
                                or "No workload data"
                            )
                        )

                    st.markdown(
                        "### 🤖 AI Weekly Summary"
                    )

                    base_summary = generate_weekly_summary(
                        stats
                    )

                    ai_summary = None

                    ai_prompt = (
                        "Write a concise employee wellness "
                        "weekly summary using ONLY these actual "
                        "stored values. Do not invent missing "
                        "values and do not diagnose the employee. "
                        + base_summary
                        + " "
                        + str(stats)
                    )

                    try:

                        ai_resp = requests.post(
                            f"{BACKEND_URL}/chat",
                            json={
                                "message": ai_prompt,
                                "history": []
                            },
                            headers=headers,
                            timeout=60
                        )

                        if ai_resp.status_code == 200:
                            ai_summary = (
                                ai_resp.json().get(
                                    "reply"
                                )
                            )

                    except requests.exceptions.RequestException:
                        ai_summary = None

                    summary = (
                        ai_summary.strip()
                        if ai_summary
                        else base_summary
                    )

                    st.info(
                        summary
                    )

                    recs = recommendations(
                        stats
                    )

                    awards = achievements(
                        stats
                    )

                    r1, r2 = st.columns(2)

                    with r1:

                        st.markdown(
                            "### 💡 Personalized Recommendations"
                        )

                        for item in recs:
                            st.write(
                                "• " + item
                            )

                    with r2:

                        st.markdown(
                            "### 🏆 Achievements"
                        )

                        for item in awards:
                            st.write(
                                item
                            )

                    st.markdown(
                        "### 📄 Download Weekly Wellness Report"
                    )

                    pdf_bytes = build_weekly_pdf(
                        user["username"],
                        user["email"],
                        report,
                        stats,
                        summary,
                        recs,
                        awards,
                        figures
                    )

                    st.download_button(
                        "Download Weekly Wellness Report PDF",
                        data=pdf_bytes,
                        file_name=(
                            f"weekly_wellness_"
                            f"{start_date}_{end_date}.pdf"
                        ),
                        mime="application/pdf",
                        type="primary",
                        use_container_width=True
                    )

                    for _, fig in figures:
                        plt.close(fig)

                st.markdown(
                    "</div>",
                    unsafe_allow_html=True
                )

            elif section == "Dashboard":

                history = get_user_mood_history(
                    user["id"],
                    limit=200
                )

                if not history:

                    st.info(
                        "No entries yet — pick a mood on Home "
                        "or write a journal entry to see your dashboard."
                    )

                else:

                    counts = {
                        label: 0
                        for label in MOOD_LABELS
                    }

                    for h in history:

                        if h["sentiment"] in counts:
                            counts[h["sentiment"]] += 1

                    c1, c2 = st.columns(2)

                    with c1:

                        st.markdown(
                            "<div class='mm-card'>",
                            unsafe_allow_html=True
                        )

                        st.write(
                            "**Mood distribution**"
                        )

                        fig = donut_chart(
                            counts
                        )

                        if fig:

                            st.pyplot(
                                fig,
                                use_container_width=False
                            )

                        else:

                            st.bar_chart(
                                counts
                            )

                        st.markdown(
                            "</div>",
                            unsafe_allow_html=True
                        )

                    with c2:

                        st.markdown(
                            "<div class='mm-card'>",
                            unsafe_allow_html=True
                        )

                        st.write(
                            "**Mood trend over time**"
                        )

                        by_date = {}

                        for h in history:

                            d = h["mood_date"]

                            by_date.setdefault(
                                d,
                                []
                            ).append(
                                MOOD_TO_NUM.get(
                                    h["sentiment"],
                                    0
                                )
                            )

                        trend = {
                            str(d):
                            sum(v) / len(v)
                            for d, v in sorted(
                                by_date.items()
                            )
                        }

                        st.line_chart(
                            trend
                        )

                        st.markdown(
                            "</div>",
                            unsafe_allow_html=True
                        )

                    st.markdown(
                        "<div class='mm-card'>",
                        unsafe_allow_html=True
                    )

                    st.write(
                        "**Emotions detected from journal entries**"
                    )

                    emo_counts = {}

                    for h in history:

                        if (
                            h["source"] == "nlp"
                            and h["emotion"]
                        ):

                            emo_counts[h["emotion"]] = (
                                emo_counts.get(
                                    h["emotion"],
                                    0
                                ) + 1
                            )

                    if emo_counts:

                        st.bar_chart(
                            emo_counts
                        )

                    else:

                        st.caption(
                            "No journal-based emotion data yet."
                        )

                    st.markdown(
                        "</div>",
                        unsafe_allow_html=True
                    )

                    st.markdown(
                        "<div class='mm-card'>",
                        unsafe_allow_html=True
                    )

                    st.write(
                        "**Recent activity**"
                    )

                    table_rows = [
                        {
                            "Date": h["mood_date"],
                            "Time": h["created_at"].strftime(
                                "%H:%M"
                            ),
                            "Mood":
                                f"{style_for(h['sentiment'])['emoji']} "
                                f"{h['sentiment']}",
                            "Confidence":
                                f"{h['confidence']:.0%}"
                                if h.get("confidence") is not None
                                else "—",
                            "Source": h["source"],
                        }
                        for h in history[:15]
                    ]

                    st.dataframe(
                        table_rows,
                        use_container_width=True
                    )

                    st.markdown(
                        "</div>",
                        unsafe_allow_html=True
                    )

                st.markdown(
                    "<div class='mm-card'>",
                    unsafe_allow_html=True
                )

                st.write(
                    "**Team mood trend (last 30 days)**"
                )

                history = get_all_employee_mood_logs(
                    limit_days=30
                )

                if not history:

                    st.info(
                        "Not enough data yet to draw a trend chart."
                    )

                else:

                    by_date = {}

                    for row in history:

                        d = row["mood_date"]

                        by_date.setdefault(
                            d,
                            []
                        ).append(
                            MOOD_TO_NUM.get(
                                row["sentiment"],
                                0
                            )
                        )

                    trend = {
                        str(d):
                        sum(v) / len(v)
                        for d, v
                        in sorted(
                            by_date.items()
                        )
                    }

                    st.line_chart(
                        trend
                    )

                    st.caption(
                        "Average mood score per day across all employees "
                        "(2 = Amazing, 1 = Happy, 0 = Normal, -1 = Sad, -2 = Angry)"
                    )

                st.markdown(
                    "</div>",
                    unsafe_allow_html=True
                )

            st.stop()



# ============================================================
# WELCOME / LOGIN SCREEN
# ============================================================
if st.session_state.page == "welcome":

    if "quote" not in st.session_state:
        st.session_state.quote = random.choice(QUOTES)

    hour = datetime.now().hour
    greeting = "Good Morning" if hour < 12 else ("Good Afternoon" if hour < 18 else "Good Evening")

    left, right = st.columns([3, 2])

    with left:
        st.markdown(
            '<div class="login-shell" style="padding:32px 36px;">'
            '<span class="login-blob-1"></span><span class="login-blob-2"></span>'
            '<span class="sparkle" style="top:20px;right:30px;animation-delay:0s">✨</span>'
            '<span class="sparkle" style="top:80px;right:70px;animation-delay:0.8s">✨</span>'
            '<span class="sparkle" style="top:140px;right:20px;animation-delay:1.6s">💫</span>',
            unsafe_allow_html=True,
        )
        st.markdown(
            f"<div class='brand-row'>✨ MoodMentor</div>",
            unsafe_allow_html=True,
        )
        st.markdown(
            f"<div class='quote-box'>\" {st.session_state.quote} \"</div>",
            unsafe_allow_html=True,
        )

        mode = st.session_state.auth_mode

        if mode == "login":
            st.markdown(f"<div class='greet-title'>{greeting},</div>", unsafe_allow_html=True)
            st.markdown("<div class='greet-sub'>Let's check in on your well-being today.</div>", unsafe_allow_html=True)

            email = st.text_input("Email Address", placeholder="Enter your email", label_visibility="collapsed", key="login_email")
            pw = st.text_input("Password", type="password", placeholder="Enter your password", label_visibility="collapsed", key="login_pw")
            keep_signed_in = st.checkbox("Keep me signed in", value=True)

            if st.button("Continue to Dashboard 🚀", type="primary", use_container_width=True):
                u = get_user(email.strip().lower())
                if not u or not check_pw(pw, u["password_hash"]):
                    st.error("Invalid email or password.")
                elif not u["is_verified"]:
                    st.warning("Verify your email first.")
                    st.session_state.email = u["email"]; goto_auth("verify")
                else:
                    st.session_state.token = make_token(u)
                    st.rerun()

            st.markdown("<div class='divider-text'>or continue with</div>", unsafe_allow_html=True)
            sc1, sc2 = st.columns(2)
            with sc1:
                if st.button("🌐 Google", use_container_width=True):
                    st.info("Google sign-in isn't connected yet — use email/password for now.")
            with sc2:
                if st.button("🍏 Apple", use_container_width=True):
                    st.info("Apple sign-in isn't connected yet — use email/password for now.")

            st.write("")
            st.markdown(
                f"<div style='text-align:center;color:{MUTED};font-weight:600'>New to MoodMentor?</div>",
                unsafe_allow_html=True,
            )
            c1, c2 = st.columns(2)
            with c1:
                if st.button("Create an account", use_container_width=True): goto_auth("signup")
            with c2:
                if st.button("Forgot password?", use_container_width=True): goto_auth("forgot")

        elif mode == "signup":
            st.markdown("<div class='greet-title'>Create Account</div>", unsafe_allow_html=True)
            st.markdown("<div class='greet-sub'>Let's get you started.</div>", unsafe_allow_html=True)
            with st.form("signup"):
                username = st.text_input("Full Name", placeholder="Enter your full name")
                email = st.text_input("Email", placeholder="Enter your email")
                pw = st.text_input("Password", type="password", placeholder="Create password")
                role_label = st.radio("I am signing up as a:", ["Employee", "Manager"], horizontal=True)
                go = st.form_submit_button("Send OTP 🚀", type="primary", use_container_width=True)
            if go:
                email = email.strip().lower()
                role = "manager" if role_label == "Manager" else "employee"
                if len(username) < 3:
                    st.error("Username too short.")
                elif not valid_pw(pw):
                    st.error("Password needs 8+ chars, letters and numbers.")
                elif username_taken(username) or get_user(email):
                    st.error("Username or email already in use.")
                else:
                    create_user(username, email, pw, role=role)
                    code = new_otp(); save_otp(email, code, "signup")
                    ok, msg = send_otp(email, code, "signup")
                    if ok:
                        st.session_state.email = email
                        st.success("Check your email for the code.")
                        goto_auth("verify")
                    else:
                        st.error(f"Email failed: {msg}")
            if st.button("Already have an account? Login"): goto_auth("login")

        elif mode == "verify":
            email = st.session_state.email
            st.markdown("<div class='greet-title'>Verify OTP</div>", unsafe_allow_html=True)
            st.markdown(f"<div class='greet-sub'>We have sent a 6-digit code to {email}</div>", unsafe_allow_html=True)
            with st.form("verify"):
                code = st.text_input("Code", max_chars=6, placeholder="Enter 6-digit code")
                go = st.form_submit_button("Verify OTP", type="primary", use_container_width=True)
            if go:
                if check_otp(email, code.strip(), "signup"):
                    verify_user(email)
                    st.success("Verified! Please log in.")
                    goto_auth("login")
                else:
                    st.error("Invalid or expired code.")
            if st.button("← Back to login"): goto_auth("login")

        elif mode == "forgot":
            st.markdown("<div class='greet-title'>🔑 Forgot password</div>", unsafe_allow_html=True)
            with st.form("forgot"):
                email = st.text_input("Your account email")
                go = st.form_submit_button("Send reset code", type="primary", use_container_width=True)
            if go:
                email = email.strip().lower()
                if get_user(email):
                    code = new_otp(); save_otp(email, code, "password_reset")
                    send_otp(email, code, "password_reset")
                st.session_state.email = email
                st.info("If that email exists, a code was sent.")
                goto_auth("reset")
            if st.button("← Back to login"): goto_auth("login")

        elif mode == "reset":
            email = st.session_state.email
            st.markdown("<div class='greet-title'>🔄 Reset password</div>", unsafe_allow_html=True)
            with st.form("reset"):
                code = st.text_input("Reset code", max_chars=6)
                pw = st.text_input("New password", type="password")
                go = st.form_submit_button("Reset", type="primary", use_container_width=True)
            if go:
                if not valid_pw(pw):
                    st.error("Password needs 8+ chars, letters and numbers.")
                elif not check_otp(email, code.strip(), "password_reset"):
                    st.error("Invalid or expired code.")
                else:
                    set_password(email, pw)
                    st.success("Password reset. Please log in.")
                    goto_auth("login")
            if st.button("← Back to login"): goto_auth("login")

        st.markdown("</div>", unsafe_allow_html=True)

    with right:
        st.markdown(
            f"""
            <div style="position:relative; border-radius:30px; overflow:hidden;
                        box-shadow:0 30px 60px -18px rgba(99,102,241,0.35); height:100%; min-height:520px;
                        animation: popIn 700ms cubic-bezier(0.22,1,0.36,1);">
                <img src="data:image/jpeg;base64,{WELCOME_IMAGE_B64}"
                     style="width:100%; height:100%; object-fit:cover; display:block;" />
                <div style="position:absolute; top:0; left:0; right:0; height:120px;
                            background:linear-gradient(180deg, rgba(236,72,153,0.28), transparent);
                            pointer-events:none;"></div>
                <div style="position:absolute; bottom:20px; left:20px; right:20px;
                            background:rgba(255,255,255,0.94); backdrop-filter:blur(14px);
                            padding:18px 20px; border-radius:20px;
                            box-shadow:0 14px 32px -10px rgba(99,102,241,0.35);
                            border: 1.5px solid rgba(255,255,255,0.9);">
                    <div class="stars-twinkle" style="color:#F59E0B; font-size:1.1rem; margin-bottom:6px;">★★★★★</div>
                    <div style="font-size:0.9rem; font-weight:500; color:#1E1B2E; line-height:1.5; margin-bottom:8px;">
                        "Mood Mentor completely changed how I track my daily habits and emotional well-being. The journaling feature is beautiful!"
                    </div>
                    <div style="display:flex;align-items:center;gap:8px;">
                        <div style="width:26px;height:26px;border-radius:50%;background:linear-gradient(135deg,#EC4899,#9F7AEA);color:white;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:800;">S</div>
                        <div style="font-size:0.82rem; font-weight:800; background:linear-gradient(135deg,#EC4899,#6366F1);-webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text;">Sarah Jenkins</div>
                    </div>
                </div>
            </div>
            """,
            unsafe_allow_html=True,
        )

    st.stop()

Writing app.py


In [11]:
# ============================================================
# CELL 6 — nlp_pipeline.py  (UNCHANGED from your original)
# ============================================================
%%writefile nlp_pipeline.py
"""
nlp_pipeline.py
Multilingual NLP pipeline for employee feedback:
normalize -> detect language -> clean -> tokenize -> stopword-filter ->
translate to English -> lemmatize -> sentiment (VADER) -> emotion (BERT).

Stopword filtering uses the `stopwordsiso` package, which ships stopword
sets for 50+ languages keyed by ISO 639-1 code (the same codes langdetect
returns), so any supported language is handled automatically instead of
needing a hardcoded list per language. If the detected language isn't in
stopwordsiso's coverage, filtering is simply skipped for that text.

Heavy libs (spacy model, translator, vader, BERT emotion model, Qwen chat
model) load once at import time via lazy module-level globals, so repeated
/analyze calls reuse them.
"""

import re
import ftfy
import emoji
import spacy
import torch
import stopwordsiso
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline as hf_pipeline,
)
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DetectorFactory.seed = 0

_nlp = None
_vader = None
_qwen_model = None
_qwen_tokenizer = None
_bert_emotion_pipeline = None

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

BERT_EMOTION_MODEL_NAME = "bhadresh-savani/bert-base-go-emotion"

LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
    "fr": "French", "de": "German", "es": "Spanish", "pt": "Portuguese",
    "ar": "Arabic", "zh": "Chinese", "ja": "Japanese", "ko": "Korean", "ru": "Russian",
}


def _get_stopwords(language_code: str) -> set:
    """
    Returns the stopword set for `language_code` using stopwordsiso, which
    covers 50+ languages by ISO 639-1 code. Returns an empty set for any
    language it doesn't cover -- filtering is skipped rather than failing,
    so unsupported languages still flow through the rest of the pipeline.
    """
    if stopwordsiso.has_lang(language_code):
        return stopwordsiso.stopwords(language_code)
    return set()

EMOTION_LABELS = ["Happy", "Sad", "Stress", "Angry", "Fear", "Neutral"]

EMOTION_EMOJI = {
    "Happy": "\U0001F60A", "Sad": "\U0001F622", "Stress": "\U0001F62B",
    "Angry": "\U0001F621", "Fear": "\U0001F628", "Neutral": "\U0001F610",
}

GOEMOTIONS_TO_APP_LABEL = {
    "joy": "Happy", "amusement": "Happy", "excitement": "Happy",
    "love": "Happy", "gratitude": "Happy", "optimism": "Happy",
    "relief": "Happy", "pride": "Happy", "admiration": "Happy",
    "approval": "Happy", "caring": "Happy",

    "sadness": "Sad", "disappointment": "Sad", "grief": "Sad",
    "remorse": "Sad",

    "nervousness": "Stress", "embarrassment": "Stress",
    "confusion": "Stress",

    "anger": "Angry", "annoyance": "Angry", "disgust": "Angry",
    "disapproval": "Angry",

    "fear": "Fear",

    "neutral": "Neutral", "realization": "Neutral", "surprise": "Neutral",
    "curiosity": "Neutral", "desire": "Neutral",
}


def _get_nlp():
    """Lazy-load the multilingual spaCy model once per process."""
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp


def _get_vader():
    global _vader
    if _vader is None:
        _vader = SentimentIntensityAnalyzer()
    return _vader


def _get_qwen():
    """Lazy-load Qwen2.5-0.5B-Instruct once per process (GPU if available).
    Still used by the wellness chatbot (wellness_chat_reply) -- only the
    emotion-detection step now uses BERT instead."""
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
    return _qwen_model, _qwen_tokenizer


def _get_bert_emotion_pipeline():
    """
    Lazy-load the fine-tuned BERT emotion classifier once per process, using
    Hugging Face's `pipeline()` helper -- this bundles the tokenizer and the
    model together so we just call it with raw text and get scores back.

    `top_k=None` tells the pipeline to return a score for every label
    instead of just the single top prediction, so we can build a full
    scores dict (matching what the UI already expects).
    """
    global _bert_emotion_pipeline
    if _bert_emotion_pipeline is None:
        _bert_emotion_pipeline = hf_pipeline(
            "text-classification",
            model=BERT_EMOTION_MODEL_NAME,
            top_k=None,
            device=0 if torch.cuda.is_available() else -1,
        )
    return _bert_emotion_pipeline


def _bert_emotion(text: str) -> dict:
    """
    Classifies `text` using the fine-tuned BERT GoEmotions model, then maps
    the 28 GoEmotions labels down to our 6 app-level EMOTION_LABELS by
    summing mapped scores. Returns the same shape the rest of the app
    already expects: {"emotion": <label>, "scores": {label: 0-1, ...}}.
    """
    classifier = _get_bert_emotion_pipeline()

    if not text.strip():
        text = "(empty feedback)"

    raw_predictions = classifier(text, truncation=True)[0]

    app_scores = {label: 0.0 for label in EMOTION_LABELS}
    for pred in raw_predictions:
        goemotion_label = pred["label"].lower()
        app_label = GOEMOTIONS_TO_APP_LABEL.get(goemotion_label, "Neutral")
        app_scores[app_label] += pred["score"]

    total = sum(app_scores.values()) or 1.0
    app_scores = {label: round(score / total, 4) for label, score in app_scores.items()}

    final_emotion = max(app_scores, key=app_scores.get)
    confidence = app_scores[final_emotion]
    return {"emotion": final_emotion, "scores": app_scores, "confidence": confidence}


def process_employee_feedback(text: str) -> dict:
    """Runs the full pipeline on a single blob of text and returns a results dict."""
    nlp = _get_nlp()
    vader = _get_vader()

    normalized_text = ftfy.fix_text(text)

    try:
        language = detect(normalized_text)
    except Exception:
        language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")

    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]

    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]

    selected_stopwords = _get_stopwords(language)
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)

    try:
        translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except Exception as error:
        translated_text = f"Translation failed: {error}"

    english_doc = nlp(translated_text)
    lemmas = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmas)

    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05:
        final_sentiment = "Positive \U0001F60A"
    elif compound_score <= -0.05:
        final_sentiment = "Negative \U0001F614"
    else:
        final_sentiment = "Neutral \U0001F610"

    bert_result = _bert_emotion(translated_text)
    emotion_scores = bert_result["scores"]
    final_emotion_label = bert_result["emotion"]
    final_emotion = f"{final_emotion_label} {EMOTION_EMOJI.get(final_emotion_label, '')}"
    emotion_confidence = bert_result["confidence"]

    return {
        "language_code": language,
        "detected_language": detected_language,
        "normalized_text": normalized_text,
        "cleaned_text": cleaned_text,
        "sentences": sentences,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": emotion_scores,
        "final_emotion": final_emotion,
        "emotion_confidence": emotion_confidence,
    }


CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die", "self harm",
    "self-harm", "hurt myself", "not worth living", "no reason to live",
]

CRISIS_MESSAGE = (
    "I'm really glad you reached out, and I want to make sure you get support "
    "beyond what I can offer here. If you're in immediate danger, please contact "
    "your local emergency number right now. You can also reach a crisis line: "
    "in India, AASRA is available at +91-9820466726 (24/7). If you're outside "
    "India, please look up a local crisis helpline or talk to a trusted person "
    "or your HR/EAP contact. You don't have to go through this alone."
)

WELLNESS_SYSTEM_PROMPT = (
    "You are a supportive workplace wellness assistant for employees. "
    "Your role is to listen, validate feelings, and offer general, gentle "
    "coping suggestions (like breathing exercises, taking a short break, "
    "or talking to a trusted colleague or manager). "
    "You are NOT a therapist or doctor: never diagnose any condition, never "
    "claim expertise you don't have, and never give medical or medication "
    "advice. If the employee describes something serious (ongoing crisis, "
    "self-harm, harming others), gently encourage them to contact a mental "
    "health professional, their HR/EAP program, or a crisis helpline. "
    "Keep replies short (2-4 sentences), warm, and non-judgmental. "
    "Avoid clinical labels and avoid being preachy or repetitive."
)


def _contains_crisis_language(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in CRISIS_KEYWORDS)


def wellness_chat_reply(message: str, history: list[dict] | None = None) -> dict:
    """
    Generates a supportive wellness chatbot reply using the Qwen chat model.
    (The chatbot still uses Qwen -- it needs to generate free-form
    conversational replies, which is a generation task, not a
    classification task, so BERT isn't a fit here.)

    `history` is an optional list of {"role": "user"|"assistant", "content": str}
    dicts representing prior turns in the conversation (kept short/recent by
    the caller — this function does not trim it).

    Always checks for crisis language first; if found, returns a fixed,
    resource-pointing message instead of an LLM-generated one, since we
    never want a small model improvising in a safety-critical moment.
    """
    if _contains_crisis_language(message):
        return {"reply": CRISIS_MESSAGE, "flagged": True}

    model, tokenizer = _get_qwen()

    messages = [{"role": "system", "content": WELLNESS_SYSTEM_PROMPT}]
    for turn in (history or []):
        if turn.get("role") in ("user", "assistant") and turn.get("content"):
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if not reply:
        reply = "I'm here and listening — could you tell me a bit more about how you're feeling?"

    return {"reply": reply, "flagged": False}

Writing nlp_pipeline.py


In [12]:
# ============================================================
# CELL 7 — backend.py  (UNCHANGED from your original)
# ============================================================
%%writefile backend.py
import os, io, jwt, csv
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback, wellness_chat_reply
load_dotenv()

SECRET = os.getenv("JWT_SECRET")
app = FastAPI(title="Upload API")

app.add_middleware(CORSMiddleware, allow_origins=["*"],
                    allow_methods=["*"], allow_headers=["*"])

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    lines = text.splitlines()
    row_count = len(lines)
    preview_lines = lines[:20]

    columns = None
    preview_rows = None
    if ext == "csv":
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if rows:
            columns = rows[0]
            preview_rows = rows[1:21]
            row_count = max(len(rows) - 1, 0)

    return {
        "filename": name,
        "type": ext,
        "uploaded_by": user["username"],
        "row_count": row_count,
        "columns": columns,
        "preview_rows": preview_rows,
        "preview_lines": None if ext == "csv" else preview_lines,
    }


def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    """
    Returns (text_blob, used_column). For TXT, used_column is None.
    For CSV, joins all non-empty values of the chosen column (or the last
    column if none/invalid was specified) into one whitespace-joined blob —
    matches the notebook's "whole file as one blob" behavior.
    """
    text = raw.decode("utf-8")

    if ext == "txt":
        return text.strip(), None

    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows:
        raise HTTPException(400, "CSV file has no rows.")

    header = rows[0]
    data_rows = rows[1:]
    if not data_rows:
        raise HTTPException(400, "CSV file has a header but no data rows.")

    col_index = None
    if column and column in header:
        col_index = header.index(column)
    else:
        col_index = len(header) - 1

    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob:
        raise HTTPException(400, f"Column '{header[col_index]}' has no readable text.")
    return blob, header[col_index]


@app.post("/analyze")
async def analyze(file: UploadFile = File(...), column: str = Form(None),
                   authorization: str = Header(None)):
    """
    Runs the multilingual NLP pipeline (language detection, cleaning,
    stopword filtering, translation, lemmatization, VADER sentiment,
    keyword-based emotion) on an uploaded .csv or .txt file.
    """
    get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text_blob, used_column = _extract_text_blob(raw, ext, column)
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    results = process_employee_feedback(text_blob)
    results["filename"] = name
    results["file_type"] = ext.upper()
    results["used_column"] = used_column
    results["original_char_count"] = len(text_blob)
    return results



class TextIn(BaseModel):
    text: str

@app.post("/analyze-text")
async def analyze_text(payload: TextIn, authorization: str = Header(None)):
    """Same NLP pipeline as /analyze, but for text typed directly into the
    Journal tab's textbox instead of an uploaded file."""
    get_user(authorization)

    text_blob = payload.text.strip()
    if not text_blob:
        raise HTTPException(400, "Text cannot be empty.")

    results = process_employee_feedback(text_blob)
    results["filename"] = None
    results["file_type"] = "TEXT"
    results["used_column"] = None
    results["original_char_count"] = len(text_blob)
    return results

class ChatTurn(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    message: str
    history: list[ChatTurn] = []


@app.post("/chat")
async def chat(payload: ChatRequest, authorization: str = Header(None)):
    """
    Wellness support chatbot endpoint. Stateless on the server: the client
    (Streamlit) sends the recent conversation history along with each new
    message, and we generate the next reply with the same Qwen model used
    for emotion detection.
    """
    get_user(authorization)

    message = payload.message.strip()
    if not message:
        raise HTTPException(400, "Message cannot be empty.")

    history = [turn.dict() for turn in payload.history]
    result = wellness_chat_reply(message, history=history)
    return result

Writing backend.py


In [15]:
import db
from dotenv import load_dotenv
import importlib, os, subprocess, time

load_dotenv(override=True)
importlib.reload(db)
db.init_db()
print("✅ PostgreSQL connected and weekly-report tables/columns are ready.")

from pyngrok import ngrok, conf
from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")

conf.get_default().auth_token = NGROK_AUTHTOKEN

# Clean up tunnels/processes created by earlier runs in this Colab runtime.
try:
    for tunnel in ngrok.get_tunnels():
        try:
            ngrok.disconnect(tunnel.public_url)
        except Exception:
            pass
except Exception:
    pass
ngrok.kill()
get_ipython().system_raw("pkill -f streamlit || true")
get_ipython().system_raw("pkill -f uvicorn || true")
time.sleep(1)

# Backend remains local; Streamlit talks to it over localhost.
get_ipython().system_raw("uvicorn backend:app --host 0.0.0.0 --port 8000 > /tmp/uvicorn.log 2>&1 &")
time.sleep(5)
os.environ["BACKEND_URL"] = "http://localhost:8000"

get_ipython().system_raw(
    "streamlit run app.py --server.port 8501 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false > /tmp/streamlit.log 2>&1 &"
)
time.sleep(5)

# Start one fresh public tunnel. No fixed ngrok domain is configured.
try:
    public_url_object = ngrok.connect(addr=8501, proto="http")
    print(f"🌐 Your app is live at: {public_url_object.public_url}")
    print("Open the URL above. Keep this Colab runtime running.")
except Exception as e:
    print("❌ ngrok could not create a new public tunnel.")
    print("If ngrok reports that an endpoint is already online, stop the older endpoint in the ngrok dashboard, then rerun this cell.")
    print(f"Details: {e}")


DEBUG: DB_HOST being used by db.py: ep-ancient-leaf-azn3k07u-pooler.c-3.ap-southeast-1.aws.neon.tech
✅ PostgreSQL connected and weekly-report tables/columns are ready.


❌ ngrok could not create a new public tunnel.
If ngrok reports that an endpoint is already online, stop the older endpoint in the ngrok dashboard, then rerun this cell.
Details: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://onward-numerate-primary.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}



In [ ]:
# ============================================================
# DIAGNOSTIC CELL — run this to check your account status
# ============================================================
from db import cursor

email_to_check = "Enter Your Email Here"  # <-- put the exact email you're trying to log in with

with cursor() as cur:
    cur.execute("SELECT id, username, email, is_verified, role, password_hash FROM users WHERE email=%s",
                (email_to_check.strip().lower(),))
    row = cur.fetchone()

if row is None:
    print("❌ No account exists with that email.")
    print("   → You need to sign up first (or check for a typo in the email).")
else:
    print("✅ Account found:")
    print(f"   username: {row['username']}")
    print(f"   email: {row['email']}")
    print(f"   is_verified: {row['is_verified']}")
    print(f"   role: {row['role']}")
    print(f"   password_hash starts with: {row['password_hash'][:10]}...")
    if not row['is_verified']:
        print("\n⚠️  This account exists but is NOT verified.")
        print("   That would normally show 'Verify your email first', not 'Invalid email or password'.")
    else:
        print("\n   Account is verified. If login still fails, the password typed doesn't match this hash.")